In [347]:
import pandas as pd
import glob
from inference import get_model_available, get_data_academic_risk, predict_academic_risk
import numpy as np

In [348]:
from dotenv import load_dotenv
load_dotenv()

True

In [349]:
import os

os.add_dll_directory('C:\\Program Files\\IBM\\SQLLIB\\BIN')
# conectar a la base de datos IBM Db2
import ibm_db

In [350]:
conn_str = 'DATABASE=' + os.getenv('DATABASE') + ';HOSTNAME=' + os.getenv('HOSTNAME') + ';PORT=' + os.getenv('PORT') + ';PROTOCOL=TCPIP;UID=' + os.getenv('USERNAME_DB') + ';PWD=' + os.getenv('PASSWORD_DB') + ';'
conn = ibm_db.connect(conn_str, '', '')
# conn = ibm_db.connect(os.getenv('DATABASE'), os.getenv('USERNAME_DB'), os.getenv('PASSWORD_DB')) # os.getenv('HOSTNAME'), os.getenv('PORT')

if conn:
    print("Conexión exitosa")
else:
    print("Error al conectar")

Conexión exitosa


In [351]:
anio_base, termino_base = 2025, 2  # Periodo actual
cod_materia_objetivo = "CCPG1056" #"CCPG1042"# "MATG1058", 'CCPG1052'  # Materia objetivo

In [352]:
def get_data_from_query(sql_str):
    stmt_select  = ibm_db.exec_immediate(conn, sql_str)
    
    # Fetch all rows
    data_list = []
    result = ibm_db.fetch_assoc(stmt_select)
    while result:
        # print(result)
        data_list.append(result)
        result = ibm_db.fetch_assoc(stmt_select)
    return data_list

In [353]:
modelo, label_encoders, feature_info = get_model_available()


📂 Cargando modelos y configuración...

✅ Modelo seleccionado: Random Forest: ../models/random_forest_model.pkl
✓ Modelo cargado desde '../models/random_forest_model.pkl'


In [354]:
def make_analysis(list_matricula, cod_materia):
    mi_df, list_student_off = get_data_academic_risk(list_matricula, cod_materia, label_encoders)

    print("Debe coinicidir la cantidad:",mi_df.shape[0] == len(list_matricula))
    if mi_df.shape[0] != len(list_matricula):
        print("⚠️ Advertencia: Algunos estudiantes no tienen datos completos para el análisis.")

    results = predict_academic_risk(modelo, feature_info, mi_df)

    stats = results['statistics']
    # print(f"📊 {stats['pred_aprobar']} estudiantes aprobarán ({stats['pct_aprobar']:.2f}%)")
    
    return results, mi_df, stats, list_student_off

### Datos de historico completo desde 2020 a 2025 2S

In [355]:
list_names_files = glob.glob("../data/riesgo_academico/all_*")
df_materias = pd.read_csv("../data/riesgo_academico/dificultad_materia.csv")

In [356]:
df_complete = pd.DataFrame()

for file in list_names_files:
    print("file", file)
    split_file = file.split("_")
    termino = split_file[-1].split(".")[0]
    if termino == "3S":
        continue
    
    df = pd.read_csv(file)

    df["anio"] = split_file[-2]
    df["termino"] = termino
    if not("MATERIA" in df.keys()):
        df = pd.merge(df, df_materias[["CODIGOMATERIA", "MATERIA"]], left_on="COD_MATERIA_ACAD_MO", right_on="CODIGOMATERIA")

    df_complete = pd.concat([df_complete, df], ignore_index=True)

file ../data/riesgo_academico\all_2020_1S.csv
file ../data/riesgo_academico\all_2020_2S.csv
file ../data/riesgo_academico\all_2021_1S.csv
file ../data/riesgo_academico\all_2021_2S.csv
file ../data/riesgo_academico\all_2022_1S.csv
file ../data/riesgo_academico\all_2022_2S.csv
file ../data/riesgo_academico\all_2023_1S.csv
file ../data/riesgo_academico\all_2023_2S.csv
file ../data/riesgo_academico\all_2024_1S.csv
file ../data/riesgo_academico\all_2024_2S.csv
file ../data/riesgo_academico\all_2025_1S.csv
file ../data/riesgo_academico\all_2025_2S.csv
file ../data/riesgo_academico\all_2026_1S.csv


C:\Users\saraujo\AppData\Local\Temp\ipykernel_80540\1209477652.py:17: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df_complete = pd.concat([df_complete, df], ignore_index=True)


In [357]:
df_complete["COD_MATERIA_ACAD_MO"].unique()

array(['ACUG1035', 'ACUG1036', 'ACUG1039', 'ACUG1040', 'ACUG1041',
       'ACUG1042', 'ACUG1043', 'ACUG1045', 'ACUG1046', 'ACUG1047',
       'ACUG1048', 'ACUG1049', 'ACUG1050', 'ACUG1051', 'ACUG1052',
       'ACUG1053', 'ACUG1054', 'ACUG1056', 'ACUG1057', 'ACUG1059',
       'ADMG1005', 'ADMG1007', 'ADMG2022', 'ADMG2024', 'ADMG2025',
       'ADMG2026', 'ADMG2028', 'ADMG2029', 'ADMG2030', 'ADMG2033',
       'ADMG2034', 'ADMG2035', 'ADMG2036', 'ADMG2037', 'ADSG1019',
       'ADSG1020', 'ADSG1021', 'ADSG1022', 'ADSG1024', 'ADSG1025',
       'ADSG1026', 'ADSG1029', 'AGRG1022', 'AGRG1023', 'AGRG1025',
       'AGRG1027', 'AGRG1028', 'AGRG1032', 'AGRG1035', 'AGRG1036',
       'AGRG1037', 'AGRG1039', 'ALIG1029', 'ALIG1030', 'ALIG1031',
       'ALIG1032', 'ALIG1033', 'ALIG1034', 'ALIG1035', 'ALIG1036',
       'ALIG1037', 'ALIG1038', 'ALIG1039', 'ALIG1041', 'ALIG1042',
       'ALIG1044', 'ALIG1045', 'ALIG1046', 'ALIG1050', 'ARQG2020',
       'ARQG2021', 'ARQG2023', 'ARQG2025', 'ARQG2031', 'ARQG20

In [358]:
df_complete["COD_ESTUDIANTE"] = df_complete["COD_ESTUDIANTE"].astype(str)

In [359]:
df_complete.shape

(366900, 30)

In [360]:
df_complete

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,FACIL,MODERADA,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL
0,201160178,ACUG1035,AP,1,87,93,"9,00","7,90",53.0,59.0,...,0,2,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN
1,201310353,ACUG1035,AP,1,85,89,"8,70","7,90",59.0,61.0,...,0,2,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN
2,201313869,ACUG1035,AP,1,86,89,"8,75","7,90",59.0,56.0,...,1,1,0,1,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN
3,201507649,ACUG1035,AP,1,83,93,"8,80","7,90",49.0,65.0,...,0,2,2,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN
4,201607884,ACUG1035,AP,1,95,95,"9,50","7,90",39.0,72.0,...,1,1,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366895,202300422,TURG2037,AC,1,0,0,"0,00","7,22",27.0,73.0,...,1,0,1,3,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,92"
366896,202104683,TURG2037,AC,1,0,0,"0,00","7,22",37.0,65.0,...,0,0,1,2,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,54"
366897,202111548,TURG2037,AC,1,0,0,"0,00","7,22",36.0,66.0,...,0,0,1,3,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,53"
366898,202103719,TURG2037,AC,1,0,0,"0,00","7,22",33.0,67.0,...,0,0,1,2,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,25"


In [361]:
# group by anio and termino
df_complete.groupby(["anio", "termino"]).size() 

anio  termino
2020  1S         35871
      2S         34086
2021  1S         32727
      2S         31286
2022  1S         29507
      2S         28859
2023  1S         28301
      2S         28195
2024  1S         28707
      2S         28488
2025  1S         30264
      2S         30609
dtype: int64

#### Carrera con datos completos de estudiantes

In [362]:
df_carreras_estudiantes = pd.read_csv("../data/riesgo_academico/carreras_estudiantes.csv")

In [363]:
df_carreras_estudiantes["CODESTUDIANTE"] = df_carreras_estudiantes["CODESTUDIANTE"].astype(str)
df_carreras_estudiantes["CODESTUDIANTE"] = df_carreras_estudiantes["CODESTUDIANTE"].str.strip()

In [364]:
# # agrupa los CODESTUDIANTE y CARRERA y cuenta la cantidad de ocurrencias. Con CODESTUDIANTE y CARRERA como columnas separadas
df_carreras_estudiantes_new = df_carreras_estudiantes.groupby(['CODESTUDIANTE', 'CARRERA', 'ANIO']).size().reset_index(name='CANTIDAD').copy()

In [365]:
# Obtener el año máximo por cada combinación de CODESTUDIANTE y CARRERA
df_carreras_estudiantes_max_anio = df_carreras_estudiantes_new.loc[
    df_carreras_estudiantes_new.groupby(['CODESTUDIANTE', 'CARRERA'])['ANIO'].idxmax()
][['CODESTUDIANTE', 'CARRERA', 'ANIO']].reset_index(drop=True)

In [366]:
# Conservar solo una fila por CODESTUDIANTE con el ANIO máximo
df_carreras_estudiantes_max_anio = (
    df_carreras_estudiantes_max_anio
    .sort_values('ANIO', ascending=False)
    .groupby('CODESTUDIANTE', as_index=False)
    .first()
)

In [367]:
df_carreras_estudiantes_max_anio

,CODESTUDIANTE,CARRERA,ANIO
0,198802423,Acuicultura,2020
1,198901423,Electricidad,2020
2,198903122,Arqueología,2024
3,198905267,Acuicultura,2020
4,199002130,Electricidad,2025
...,...,...,...
19694,202590329,Movilidad Nacional,2025
19695,202590337,Movilidad Nacional,2025
19696,202590345,Movilidad Nacional,2025
19697,202590352,Movilidad Nacional,2025


In [368]:
df_carreras_estudiantes_max_anio["CODESTUDIANTE"].value_counts()

CODESTUDIANTE
202590360    1
198802423    1
198901423    1
202590204    1
202590196    1
            ..
199500075    1
199202169    1
199201526    1
199002130    1
198905267    1
Name: count, Length: 19699, dtype: int64

In [369]:
df_carreras_estudiantes_max_anio[df_carreras_estudiantes_max_anio["CODESTUDIANTE"] == "201908803"]

,CODESTUDIANTE,CARRERA,ANIO
8433,201908803,Alimentos,2025


In [370]:
df_complete.shape, df_carreras_estudiantes_max_anio.shape

((366900, 30), (19699, 3))

In [371]:
# agregar a df_complete CARRERA desde df_carreras_estudiantes haciendo merge con COD_ESTUDIANTE de df_complete y CODESTUDIANTE de df_carreras_estudiantes
df_complete = pd.merge(df_complete, df_carreras_estudiantes_max_anio[['CODESTUDIANTE', 'CARRERA']], left_on='COD_ESTUDIANTE', right_on='CODESTUDIANTE', how='inner')
df_complete = df_complete.drop(columns=['CODESTUDIANTE'])

In [372]:
df_complete["termino_num"] = df_complete["termino"].map({"1S": 1, "2S": 2, "3S": 3})
df_complete['DIFICULTAD_MO'] = df_complete['DIFICULTAD_MO'].str.replace(',', '.').astype(float)
df_complete['PROM_MAT_REPROBADAS1'] = df_complete['PROM_MAT_REPROBADAS1'].str.replace(',', '.').astype(float)


In [373]:
df_complete

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num
0,201160178,ACUG1035,AP,1,87,93,"9,00",7.90,53.0,59.0,...,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura,1
1,201310353,ACUG1035,AP,1,85,89,"8,70",7.90,59.0,61.0,...,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura,1
2,201313869,ACUG1035,AP,1,86,89,"8,75",7.90,59.0,56.0,...,0,1,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura,1
3,201507649,ACUG1035,AP,1,83,93,"8,80",7.90,49.0,65.0,...,2,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura,1
4,201607884,ACUG1035,AP,1,95,95,"9,50",7.90,39.0,72.0,...,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366895,202300422,TURG2037,AC,1,0,0,"0,00",7.22,27.0,73.0,...,1,3,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,92",Turismo,2
366896,202104683,TURG2037,AC,1,0,0,"0,00",7.22,37.0,65.0,...,1,2,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,54",Turismo,2
366897,202111548,TURG2037,AC,1,0,0,"0,00",7.22,36.0,66.0,...,1,3,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,53",Turismo,2
366898,202103719,TURG2037,AC,1,0,0,"0,00",7.22,33.0,67.0,...,1,2,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,25",Turismo,2


#### GPA

In [374]:
list_gpa_gemeral = glob.glob("../data/riesgo_academico/gpa_*S.csv")

In [375]:
df_gpa_general = pd.DataFrame()
for file_gpa_gen in list_gpa_gemeral:
    print(file_gpa_gen)
    df_tmp = pd.read_csv(file_gpa_gen)
    df_gpa_general = pd.concat([df_gpa_general, df_tmp], ignore_index=True)

../data/riesgo_academico\gpa_general_2020_0S.csv
../data/riesgo_academico\gpa_general_2020_1S.csv
../data/riesgo_academico\gpa_general_2020_2S.csv
../data/riesgo_academico\gpa_general_2021_1S.csv
../data/riesgo_academico\gpa_general_2021_2S.csv
../data/riesgo_academico\gpa_general_2022_1S.csv
../data/riesgo_academico\gpa_general_2022_2S.csv
../data/riesgo_academico\gpa_general_2023_1S.csv
../data/riesgo_academico\gpa_general_2023_2S.csv
../data/riesgo_academico\gpa_general_2024_1S.csv
../data/riesgo_academico\gpa_general_2024_2S.csv
../data/riesgo_academico\gpa_general_2025_1S.csv
../data/riesgo_academico\gpa_general_2025_2S.csv


In [376]:
df_gpa_general['COD_ESTUDIANTE'] = df_gpa_general['COD_ESTUDIANTE'].astype(str).str.strip()

In [377]:
df_gpa_general = df_gpa_general.drop_duplicates()

In [378]:
df_gpa_general

,COD_ESTUDIANTE,ANIO,TERMINO,DENOMINADOR,NUMERADOR
0,200818094,2020,2S,4.0,32.4
1,200110245,2020,2S,6.0,36.6
2,202010070,2020,2S,6.0,46.5
3,202010120,2020,2S,6.0,40.5
4,202010625,2020,2S,6.0,39.0
...,...,...,...,...,...
99823,202414041,2025,2S,NaN,NaN
99824,202501375,2025,2S,NaN,NaN
99825,202505467,2025,2S,NaN,NaN
99826,202509501,2025,2S,NaN,NaN


#### Socioeconomico del estudiante objetivo

In [379]:
df_socioeconomico = pd.read_csv("../data/riesgo_academico/socioeconomico_17944.csv")
df_socioeconomico['CODESTUDIANTE'] = df_socioeconomico['CODESTUDIANTE'].astype(str).str.strip()

C:\Users\saraujo\AppData\Local\Temp\ipykernel_80540\1392310176.py:1: DtypeWarning: Columns (10,41,42,66,123) have mixed types. Specify dtype option on import or set low_memory=False.
  df_socioeconomico = pd.read_csv("../data/riesgo_academico/socioeconomico_17944.csv")


In [380]:
df_socioeconomico["GASTOS_RUBRO"] = ((df_socioeconomico["ALIMENTACION"] + df_socioeconomico["TRANSPORTE"] + df_socioeconomico["SERVICIOS"] +
  df_socioeconomico['ARRIENDO'] + df_socioeconomico['ALICUOTAS'] + df_socioeconomico['VESTIMENTA'] + df_socioeconomico['SALUD'] + df_socioeconomico['EDUCACION'] +
  df_socioeconomico['TARJETACREDITO'] + df_socioeconomico['ENTRETENIMIENTO'] + df_socioeconomico['OTROS']) / df_socioeconomico["NUMEROSFAMILIARES"]).round(2)


In [381]:
# eliminar duplicados de CODESTUDIANTE dejando la primera ocurrencia
df_socioeconomico = df_socioeconomico.drop_duplicates(subset=['CODESTUDIANTE'], keep='first')

### Estudiantes que estan viendo actualmente la materia objetivo

In [382]:
dir_estudiantes_viendo = '../database/planificacion_aprobacion/estudiantes_actualmente_viendo.sql'


In [383]:
with open(dir_estudiantes_viendo, 'r', encoding='utf-8') as file:
    sql_estudiantes_viendo_base = file.read()

In [384]:
sql_estudiantes_viendo = sql_estudiantes_viendo_base.split("------------")[0]
sql_estudiantes_viendo = sql_estudiantes_viendo.replace('\n', ' ')
sql_estudiantes_viendo = sql_estudiantes_viendo.replace('TT', f"{termino_base}S")
sql_estudiantes_viendo = sql_estudiantes_viendo.replace('AAAA', f"{anio_base}")
sql_estudiantes_viendo = sql_estudiantes_viendo.replace('xxxxx', f"{cod_materia_objetivo}")

In [385]:
sql_estudiantes_viendo

"SELECT COD_ESTUDIANTE, COD_MATERIA_ACAD, tm.NOMBRE, tpa.nombre carrera, hd.*  FROM espol.HISTORIA_ANIO  hd  INNER JOIN espol.TBL_MATERIA tm ON hd.COD_MATERIA_ACAD =tm.CODIGOMATERIA  INNER JOIN espol.TBL_PROGRAMA_ACADEMICO tpa ON tpa.CODCARRERA =hd.COD_CARRERA and tpa.CODDIVISION =hd.COD_DIVISION AND tpa.CODESPECIALIZ =hd.COD_ESPECIALIZ WHERE tm.codigomateria IN ('CCPG1056') AND termino='2S' AND ANIO='2025' AND hd.ESTADO_MAT_TOMADA <>'AN' ;  "

In [386]:
# Fetch all rows
data_list = get_data_from_query(sql_estudiantes_viendo)

In [387]:
df_estudiantes_viendo = pd.DataFrame(data_list)[["COD_ESTUDIANTE", "NOMBRE", "CARRERA", "COD_MATERIA_ACAD", "ANIO", "TERMINO", "VEZ_TOMADA"]].copy()

In [388]:
# hacer antes el copy porque si no no sabe si es en el copy o en el original
df_estudiantes_viendo["COD_MATERIA_ACAD"] = df_estudiantes_viendo["COD_MATERIA_ACAD"].astype(str)
df_estudiantes_viendo["COD_MATERIA_ACAD"] = df_estudiantes_viendo["COD_MATERIA_ACAD"].str.strip() 

In [389]:
df_estudiantes_viendo.shape

(65, 7)

In [390]:
df_estudiantes_viendo.head(8)

,COD_ESTUDIANTE,NOMBRE,CARRERA,COD_MATERIA_ACAD,ANIO,TERMINO,VEZ_TOMADA
0,202305462,SISTEMAS OPERATIVOS,Computación,CCPG1056,2025,2S,1
1,202308094,SISTEMAS OPERATIVOS,Computación,CCPG1056,2025,2S,1
2,202210688,SISTEMAS OPERATIVOS,Computación,CCPG1056,2025,2S,1
3,202107652,SISTEMAS OPERATIVOS,Computación,CCPG1056,2025,2S,1
4,202308953,SISTEMAS OPERATIVOS,Computación,CCPG1056,2025,2S,1
5,202109757,SISTEMAS OPERATIVOS,Computación,CCPG1056,2025,2S,1
6,202202040,SISTEMAS OPERATIVOS,Computación,CCPG1056,2025,2S,1
7,202201679,SISTEMAS OPERATIVOS,Computación,CCPG1056,2025,2S,1


In [391]:
df_estudiantes_viendo["CARRERA"].value_counts()

CARRERA
Computación    65
Name: count, dtype: int64

### Prerequisitos necesarios de la materia objetivo

In [392]:
dir_pre_co_requisitos = '../database/planificacion_aprobacion\\corequisito_prerequisito_materias.sql'

In [393]:
# queries SQL parameters
with open(dir_pre_co_requisitos, 'r', encoding='utf-8') as file:
    sql_pre_co_requisitos_base = file.read()


In [394]:
sql_pre_co_requisitos = sql_pre_co_requisitos_base.split("------------")[0]
sql_pre_co_requisitos = sql_pre_co_requisitos.replace('\n', ' ')
sql_pre_co_requisitos = sql_pre_co_requisitos.replace('xxxxx', f"{cod_materia_objetivo}")

In [395]:
sql_pre_co_requisitos

"SELECT distinct pa.nombre carrera, m.codigomateria, m.nombre materia , mm.nivel, m.HORASDOCENCIASEM,m.HORASPRACTICASSEM, m.HORASAUTONOMSEM, m.nombreingles , CASE when m.tipomateria = 'T' then 'teórica' when m.tipomateria = 'I' then 'importada' when m.tipomateria = 'p' then 'practica' when m.tipomateria = 'r' then 'teóricoPractica' when m.tipomateria = 'n' then 'nivelcero' when m.tipomateria = 'l' then 'con laboratorio' when m.tipomateria = 'a' then 'laboratorio' when m.tipomateria = 'g' then 'graduacion' when m.tipomateria = 's' then 'teórico práctico sin laboratorio' when m.tipomateria = 'm' then 'modular' when m.tipomateria = 'c' then 'mención' when m.tipomateria = 'z' then 'paralelo practico unido a teorico x codigomateria'  when m.tipomateria = 'o' then 'integradora' when m.tipomateria = 'v' then 'investigacion' else m.tipomateria end tipomateria, tc.nombre tipocredito , case when m.clasifmateria = 'g' then 'general' when m.clasifmateria = 'c' then 'complementaria' when m.clasifma

In [396]:
# Fetch all rows
data_list = get_data_from_query(sql_pre_co_requisitos)

In [397]:
df_pre_requisito = pd.DataFrame(data_list)[["CARRERA", "MATERIA_REQUISITO", "CODIGOMATERIA", "TIPO", "TIPOMATERIA", "MATERIA"]].copy()

In [398]:
df_pre_requisito = df_pre_requisito[df_pre_requisito["TIPO"] == "PR"]

In [399]:
df_pre_requisito.shape

(1, 6)

In [400]:
df_pre_requisito

,CARRERA,MATERIA_REQUISITO,CODIGOMATERIA,TIPO,TIPOMATERIA,MATERIA
0,Computación,ORGANIZACIÓN DE COMPUTADORES,CCPG1049,PR,teórica,SISTEMAS OPERATIVOS


In [401]:
materia_objetivo = df_pre_requisito["MATERIA"].iloc[0]

### Estudiantes de prerequisitos, cuales han aprobado para considerar en la planificacion

In [402]:
lis_carreras_pre = df_pre_requisito["CARRERA"].unique().tolist()

In [403]:
lis_carreras_pre

['Computación']

In [404]:
lis_cod_materias_pre = [i.strip() for i in df_pre_requisito["CODIGOMATERIA"].unique().tolist()]

In [405]:
lis_cod_materias_pre

['CCPG1049']

In [406]:
sql_estudiantes_viendo_pre = sql_estudiantes_viendo_base.split("------------")[0]
sql_estudiantes_viendo_pre = sql_estudiantes_viendo_pre.replace('\n', ' ')
sql_estudiantes_viendo_pre = sql_estudiantes_viendo_pre.replace('TT', f"{termino_base}S")
sql_estudiantes_viendo_pre = sql_estudiantes_viendo_pre.replace('AAAA', f"{anio_base}")
sql_estudiantes_viendo_pre = sql_estudiantes_viendo_pre.replace('xxxxx', "','".join(lis_cod_materias_pre))

In [407]:
sql_estudiantes_viendo_pre

"SELECT COD_ESTUDIANTE, COD_MATERIA_ACAD, tm.NOMBRE, tpa.nombre carrera, hd.*  FROM espol.HISTORIA_ANIO  hd  INNER JOIN espol.TBL_MATERIA tm ON hd.COD_MATERIA_ACAD =tm.CODIGOMATERIA  INNER JOIN espol.TBL_PROGRAMA_ACADEMICO tpa ON tpa.CODCARRERA =hd.COD_CARRERA and tpa.CODDIVISION =hd.COD_DIVISION AND tpa.CODESPECIALIZ =hd.COD_ESPECIALIZ WHERE tm.codigomateria IN ('CCPG1049') AND termino='2S' AND ANIO='2025' AND hd.ESTADO_MAT_TOMADA <>'AN' ;  "

In [408]:
# Fetch all rows
data_list = get_data_from_query(sql_estudiantes_viendo_pre)

In [409]:
df_estudiantes_viendo_pre = pd.DataFrame(data_list)[["COD_ESTUDIANTE", "COD_MATERIA_ACAD", "NOMBRE", "CARRERA", "ANIO", "TERMINO", "VEZ_TOMADA"]].copy()

In [410]:
# hacer antes el copy porque si no no sabe si es en el copy o en el original
df_estudiantes_viendo_pre["COD_MATERIA_ACAD"] = df_estudiantes_viendo_pre["COD_MATERIA_ACAD"].str.strip() 

In [411]:
"Tamaño coincide", df_estudiantes_viendo_pre["COD_ESTUDIANTE"].shape[0] == len(data_list)

('Tamaño coincide', True)

In [412]:
df_estudiantes_viendo_pre = df_estudiantes_viendo_pre[df_estudiantes_viendo_pre["CARRERA"].isin(lis_carreras_pre)]

In [413]:
df_estudiantes_viendo_pre["CARRERA"].value_counts()

CARRERA
Computación    89
Name: count, dtype: int64

In [414]:
df_estudiantes_viendo_pre["NOMBRE"].value_counts()

NOMBRE
ORGANIZACIÓN DE COMPUTADORES    89
Name: count, dtype: int64

In [415]:
df_estudiantes_viendo_pre

,COD_ESTUDIANTE,COD_MATERIA_ACAD,NOMBRE,CARRERA,ANIO,TERMINO,VEZ_TOMADA
0,202215588,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1
1,202306023,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1
2,202313326,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1
3,202407797,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1
4,202312450,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1
...,...,...,...,...,...,...,...
84,202212841,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1
85,201915410,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1
86,202312344,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1
87,202316253,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1


In [416]:
# Estudiantes que no deberian estar nuevamente si se quedan por tercera
list_tercera = df_estudiantes_viendo_pre[df_estudiantes_viendo_pre["VEZ_TOMADA"] > 2]["COD_MATERIA_ACAD"]
list_tercera

Series([], Name: COD_MATERIA_ACAD, dtype: object)

In [417]:
dict_results = {}
for cod_materia in lis_cod_materias_pre:
    list_matricula = df_estudiantes_viendo_pre[df_estudiantes_viendo_pre["COD_MATERIA_ACAD"] == cod_materia]["COD_ESTUDIANTE"].tolist()
    results, mi_df, stats, list_student_off = make_analysis(list_matricula, cod_materia)
    dict_results[cod_materia] = {
        "results": results,
        "dataframe": mi_df,
        "statistics": stats
    }


file_dir ../data/riesgo_academico/saved/inference_data.csv

✅ CCPG1049 encontrado!
   Valor original: CCPG1049, Valor encoded: 155
📊 Matrículas encontradas (89): ['202215588', '202306023', '202313326', '202407797', '202312450', '202312179', '202409645', '202315297', '202312583', '202315248', '202211728', '202208195', '202408845', '202408191', '202307773', '202208518', '202318325', '202407334', '202409777', '202308318', '202409512', '202208286', '202305827', '202315644', '202310942', '202214532', '202209243', '202408472', '202109997', '202306403', '202304374', '202211082', '202211157', '202312674', '202313078', '202313292', '202211108', '202402681', '202312823', '202216172', '202009742', '202317103', '202307799', '202306478', '202314472', '202318374', '202401691', '202212833', '202308961', '202200291', '202212890', '202302766', '202306452', '201711363', '202109617', '202311015', '202302808', '202311346', '202306965', '202213013', '202205233', '202309811', '202205266', '202107322', '2022

c:\Users\saraujo\Documents\Riesgo academico\Codigos_riesgo_academico\planificacion_aprobacion\inference.py:148: FutureWarning: Categorical.to_list is deprecated and will be removed in a future version. Use obj.tolist() instead
  results["CATEGORIA_RIESGO"] = tmp_categoria_riesgo.to_list()


In [418]:
dict_results.keys()

dict_keys(['CCPG1049'])

In [419]:
df_estudiantes_viendo_pre_tmp = df_estudiantes_viendo_pre.copy()
df_estudiantes_viendo_pre_tmp["APROBADO"] = 0

In [420]:
for cod_materia in lis_cod_materias_pre:
    list_matricula = df_estudiantes_viendo_pre[df_estudiantes_viendo_pre["COD_MATERIA_ACAD"] == cod_materia]["COD_ESTUDIANTE"].tolist()
    list_result = dict_results[cod_materia]["results"]["predictions"]
    indice_result = 0
    for i in list_matricula:
        df_estudiantes_viendo_pre_tmp.loc[(df_estudiantes_viendo_pre_tmp["COD_ESTUDIANTE"] == i) & (df_estudiantes_viendo_pre_tmp["COD_MATERIA_ACAD"] == cod_materia), "APROBADO"] = list_result[indice_result]
        indice_result += 1  

In [421]:
df_estudiantes_viendo_pre.shape, df_estudiantes_viendo_pre_tmp.shape, df_estudiantes_viendo_pre_tmp["APROBADO"].value_counts()

((89, 7),
 (89, 8),
 APROBADO
 1    86
 0     3
 Name: count, dtype: int64)

In [422]:
df_estudiantes_viendo_pre_tmp

,COD_ESTUDIANTE,COD_MATERIA_ACAD,NOMBRE,CARRERA,ANIO,TERMINO,VEZ_TOMADA,APROBADO
0,202215588,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1
1,202306023,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1
2,202313326,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1
3,202407797,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1
4,202312450,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1
...,...,...,...,...,...,...,...,...
84,202212841,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1
85,201915410,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1
86,202312344,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1
87,202316253,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1


### Analizar los estudiantes actuales (2025-2S) con el modelo actual, para quedarnos con los reprobados para la planificacion 2026


In [423]:
list_matricula = df_estudiantes_viendo["COD_ESTUDIANTE"].tolist()
len(list_matricula), df_estudiantes_viendo["COD_ESTUDIANTE"].nunique()

(65, 65)

In [424]:
df_estudiantes_viendo_tmp = df_estudiantes_viendo.copy()
df_estudiantes_viendo_tmp["APROBADO"] = 0

In [425]:
results, mi_df, stats, list_student_off = make_analysis(list_matricula, cod_materia_objetivo)

file_dir ../data/riesgo_academico/saved/inference_data.csv

✅ CCPG1056 encontrado!
   Valor original: CCPG1056, Valor encoded: 161
📊 Matrículas encontradas (65): ['202305462', '202308094', '202210688', '202107652', '202308953', '202109757', '202202040', '202201679', '202305801', '202306528', '202208260', '202306437', '202302790', '202310660', '202200473', '202306213', '202302774', '202104485', '202011086', '202311171', '202307070', '202313102', '202112033', '202207460', '202007910', '202103859', '202002705', '202107637', '202211470', '201605847', '202312534', '201405946', '202207700', '202106555', '202202578', '202009262', '202013116', '202212817', '202213005', '202106563', '202100657', '202011888', '202108924', '202305892', '202105177', '201906054', '201907953', '201611902', '201808094', '201912318', '202109823', '202209037', '202013181', '202207726', '202202131', '202002028', '202312088', '202005856', '201805397', '202004644', '202001939', '202111597', '202207890', '201901295', '2022

c:\Users\saraujo\Documents\Riesgo academico\Codigos_riesgo_academico\planificacion_aprobacion\inference.py:148: FutureWarning: Categorical.to_list is deprecated and will be removed in a future version. Use obj.tolist() instead
  results["CATEGORIA_RIESGO"] = tmp_categoria_riesgo.to_list()


In [426]:
len(results["predictions"])

65

In [427]:
index_matricula = 0
for i in list_matricula:
    if i in list_student_off:
        # no agregar a ningun lado
        # df_estudiantes_viendo_tmp.loc[(df_estudiantes_viendo_tmp["COD_ESTUDIANTE"] == i), "APROBADO"] = "OFF"
        continue
    # print("Procesando estudiante:", i, "Índice:", index_matricula)
    df_estudiantes_viendo_tmp.loc[(df_estudiantes_viendo_tmp["COD_ESTUDIANTE"] == i), "APROBADO"] = results["predictions"][index_matricula]
    index_matricula += 1

In [428]:
df_estudiantes_viendo.shape, df_estudiantes_viendo_tmp.shape, df_estudiantes_viendo_tmp["APROBADO"].value_counts()

((65, 7),
 (65, 8),
 APROBADO
 1    45
 0    20
 Name: count, dtype: int64)

### Estudiantes que ya han visto las prerequisitos pero no la objetivo y la materia objetivo. Y les tocaria ver la materia objetivo en 2026-1S

In [429]:
df_estudiantes_viendo_pre_tmp

,COD_ESTUDIANTE,COD_MATERIA_ACAD,NOMBRE,CARRERA,ANIO,TERMINO,VEZ_TOMADA,APROBADO
0,202215588,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1
1,202306023,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1
2,202313326,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1
3,202407797,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1
4,202312450,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1
...,...,...,...,...,...,...,...,...
84,202212841,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1
85,201915410,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1
86,202312344,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1
87,202316253,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,Computación,2025,2S,1,1


In [430]:
df_estudiantes_viendo_tmp

,COD_ESTUDIANTE,NOMBRE,CARRERA,COD_MATERIA_ACAD,ANIO,TERMINO,VEZ_TOMADA,APROBADO
0,202305462,SISTEMAS OPERATIVOS,Computación,CCPG1056,2025,2S,1,1
1,202308094,SISTEMAS OPERATIVOS,Computación,CCPG1056,2025,2S,1,1
2,202210688,SISTEMAS OPERATIVOS,Computación,CCPG1056,2025,2S,1,1
3,202107652,SISTEMAS OPERATIVOS,Computación,CCPG1056,2025,2S,1,1
4,202308953,SISTEMAS OPERATIVOS,Computación,CCPG1056,2025,2S,1,1
...,...,...,...,...,...,...,...,...
60,202001939,SISTEMAS OPERATIVOS,Computación,CCPG1056,2025,2S,1,1
61,202111597,SISTEMAS OPERATIVOS,Computación,CCPG1056,2025,2S,2,1
62,202207890,SISTEMAS OPERATIVOS,Computación,CCPG1056,2025,2S,1,0
63,201901295,SISTEMAS OPERATIVOS,Computación,CCPG1056,2025,2S,2,1


In [431]:
lis_cod_materias_pre, lis_carreras_pre, df_pre_requisito["CARRERA"].unique().tolist(), cod_materia_objetivo

(['CCPG1049'], ['Computación'], ['Computación'], 'CCPG1056')

In [432]:
df_pre_requisito

,CARRERA,MATERIA_REQUISITO,CODIGOMATERIA,TIPO,TIPOMATERIA,MATERIA
0,Computación,ORGANIZACIÓN DE COMPUTADORES,CCPG1049,PR,teórica,SISTEMAS OPERATIVOS


##### Descartar los RP y que estan viendo por tercera vez porque ya perdieron la carrera


In [433]:
list_tercera_and_rp = df_estudiantes_viendo_pre_tmp[(df_estudiantes_viendo_pre_tmp["APROBADO"] == 0) & (df_estudiantes_viendo_pre_tmp["VEZ_TOMADA"] > 2)]["COD_ESTUDIANTE"].unique().tolist()
df_estudiantes_viendo_tmp = df_estudiantes_viendo_tmp[~df_estudiantes_viendo_tmp["COD_ESTUDIANTE"].isin(list_tercera_and_rp)]

In [434]:
df_estudiantes_viendo_tmp.shape, df_estudiantes_viendo_tmp["APROBADO"].value_counts()

((65, 8),
 APROBADO
 1    45
 0    20
 Name: count, dtype: int64)

In [435]:
df_complete["termino"].value_counts()

termino
1S    185377
2S    181523
Name: count, dtype: int64

##### Actualizar la columna ESTADO_MAT_TOMADA_MO en df_complete

In [436]:
# materias de pre requisito, periodo actual, estudiantes viendo esas materias
df_complete[(df_complete["COD_MATERIA_ACAD_MO"].isin(lis_cod_materias_pre)) & (df_complete["anio"] == str(anio_base)) & (df_complete["termino"] == str(termino_base)+"S") & (df_complete["COD_ESTUDIANTE"].isin(df_estudiantes_viendo_pre_tmp["COD_ESTUDIANTE"]))]

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num
342551,201812005,CCPG1049,AC,1,0,0,"0,00",7.0,46.0,64.0,...,1,3,NaN,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,2025,2S,"7,46",Computación,2
342552,201915410,CCPG1049,AC,1,0,0,"0,00",7.0,40.0,64.0,...,1,4,NaN,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,2025,2S,"7,03",Computación,2
342553,202212833,CCPG1049,AC,1,0,0,"0,00",7.0,25.0,62.0,...,1,3,NaN,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,2025,2S,"7,51",Computación,2
342554,202208195,CCPG1049,AC,1,0,0,"0,00",7.0,25.0,79.0,...,0,2,NaN,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,2025,2S,"8,04",Computación,2
342555,202312179,CCPG1049,AC,1,0,0,"0,00",7.0,19.0,83.0,...,2,3,NaN,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,2025,2S,"8,41",Computación,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
342635,202205241,CCPG1049,AC,1,0,0,"0,00",7.0,28.0,65.0,...,1,3,NaN,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,2025,2S,"7,85",Computación,2
342636,202211108,CCPG1049,AC,1,0,0,"0,00",7.0,29.0,65.0,...,0,5,NaN,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,2025,2S,"7,77",Computación,2
342637,202313078,CCPG1049,AC,1,0,0,"0,00",7.0,18.0,75.0,...,1,5,NaN,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,2025,2S,"7,54",Computación,2
342638,202315644,CCPG1049,AC,1,0,0,"0,00",7.0,19.0,82.0,...,0,5,NaN,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,2025,2S,"8,33",Computación,2


In [437]:
# Mapeo de valores de APROBADO
mapping = {1: 'AP', 0: 'RP'}

In [438]:
# Actualizar df_complete
for index, row in df_estudiantes_viendo_pre_tmp.iterrows():
    # print("Procesando estudiante:", row['COD_ESTUDIANTE'], "Materia:", row['COD_MATERIA_ACAD'], "Aprobado:", row['APROBADO'])
    df_complete.loc[(df_complete['COD_ESTUDIANTE'] == row['COD_ESTUDIANTE']) & 
                    (df_complete['COD_MATERIA_ACAD_MO'] == row['COD_MATERIA_ACAD']) & 
                    (df_complete['anio'] == str(anio_base)) & 
                    (df_complete['termino'] == str(termino_base)+"S"), 
                    'ESTADO_MAT_TOMADA_MO'] = mapping[row['APROBADO']]  

In [439]:
# df_complete[(df_complete["COD_ESTUDIANTE"] == '202515714') & (df_complete["COD_MATERIA_ACAD_MO"] == 'CCPG1043')]

In [440]:
# Actualizar df_complete
for index, row in df_estudiantes_viendo_tmp.iterrows():
    # print("Procesando estudiante:", row['COD_ESTUDIANTE'], "Materia:", row['COD_MATERIA_ACAD'], "Aprobado:", row['APROBADO'])
    df_complete.loc[(df_complete['COD_ESTUDIANTE'] == row['COD_ESTUDIANTE']) & 
                    (df_complete['COD_MATERIA_ACAD_MO'] == row['COD_MATERIA_ACAD']) & 
                    (df_complete['anio'] == str(anio_base)) & 
                    (df_complete['termino'] == str(termino_base)+"S"), 
                    'ESTADO_MAT_TOMADA_MO'] = mapping[row['APROBADO']]  

In [441]:
# df_complete[(df_complete["COD_ESTUDIANTE"] == '202400701') & (df_complete["COD_MATERIA_ACAD_MO"] == 'MATG1058')]

##### Obtener los estudiantes que veran la materia objetivo en 2026-1S

In [442]:
df_complete

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num
0,201160178,ACUG1035,AP,1,87,93,"9,00",7.90,53.0,59.0,...,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura,1
1,201310353,ACUG1035,AP,1,85,89,"8,70",7.90,59.0,61.0,...,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura,1
2,201313869,ACUG1035,AP,1,86,89,"8,75",7.90,59.0,56.0,...,0,1,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura,1
3,201507649,ACUG1035,AP,1,83,93,"8,80",7.90,49.0,65.0,...,2,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura,1
4,201607884,ACUG1035,AP,1,95,95,"9,50",7.90,39.0,72.0,...,1,0,NaN,ACUG1035,ACUICULTURA ORNAMENTAL,2020,1S,NaN,Acuicultura,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366895,202300422,TURG2037,AC,1,0,0,"0,00",7.22,27.0,73.0,...,1,3,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,92",Turismo,2
366896,202104683,TURG2037,AC,1,0,0,"0,00",7.22,37.0,65.0,...,1,2,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,54",Turismo,2
366897,202111548,TURG2037,AC,1,0,0,"0,00",7.22,36.0,66.0,...,1,3,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,53",Turismo,2
366898,202103719,TURG2037,AC,1,0,0,"0,00",7.22,33.0,67.0,...,1,2,NaN,TURG2037,OPERACIÓN TURÍSTICA,2025,2S,"7,25",Turismo,2


#### 1. Repitentes: Estudiantes con último estado RP o PF en la materia objetivo

In [443]:
# Filtrar estudiantes que han cursado la materia objetivo
df_cursaron_objetivo = df_complete[df_complete['COD_MATERIA_ACAD_MO'] == cod_materia_objetivo].copy()

# Verificar si alguna vez aprobaron (AP) la materia objetivo
estudiantes_con_ap = df_cursaron_objetivo[df_cursaron_objetivo['ESTADO_MAT_TOMADA_MO'] == 'AP']['COD_ESTUDIANTE'].unique()

# Filtrar solo estudiantes que nunca aprobaron (excluir los que tienen AP)
df_sin_ap = df_cursaron_objetivo[~df_cursaron_objetivo['COD_ESTUDIANTE'].isin(estudiantes_con_ap)]

print("Cantidad de estudiantes que cursaron la materia objetivo mas de dos vez sin aprobar:", df_sin_ap['VEZ_TOMADA_MO'].value_counts()[2:].sum())

# Filtrar solo RP o PF y han visto la materia mas de dos veces VEZ_TOMADA_MO
df_rp_pf = df_sin_ap[(df_sin_ap['ESTADO_MAT_TOMADA_MO'].isin(['RP', 'PF'])) & (df_sin_ap['VEZ_TOMADA_MO'] < 3)].copy()
print("Deberian ser igual: ", df_sin_ap.shape[0] == df_rp_pf.shape[0])

# Ordenar por estudiante y fecha para obtener el último registro
df_rp_pf = df_rp_pf.sort_values(['anio', 'termino_num'], ascending=[True, True])

# Obtener el último registro por estudiante
df_repitentes = df_rp_pf.groupby('COD_ESTUDIANTE').first().reset_index()

Cantidad de estudiantes que cursaron la materia objetivo mas de dos vez sin aprobar: 4
Deberian ser igual:  False


In [444]:
df_repitentes.shape

(46, 32)

In [445]:
print(f"Total de repitentes: {df_repitentes.shape[0]}")
df_repitentes.head()

Total de repitentes: 46


,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num
0,200123669,CCPG1056,RP,1,21,20,"2,05",5.72,10.0,61.0,...,1,2,None,CCPG1056,SISTEMAS OPERATIVOS,2021,2S,None,Computación,2
1,200525939,CCPG1056,RP,1,22,2,"1,20",5.72,18.0,64.0,...,0,1,None,CCPG1056,SISTEMAS OPERATIVOS,2021,1S,None,Computación,1
2,200533511,CCPG1056,RP,1,4,0,"0,20",5.72,10.0,61.0,...,0,4,None,CCPG1056,SISTEMAS OPERATIVOS,2024,2S,None,Computación,2
3,201020203,CCPG1056,RP,1,12,0,"0,60",5.72,55.0,57.0,...,0,1,None,CCPG1056,SISTEMAS OPERATIVOS,2021,2S,None,Computación,2
4,201122213,CCPG1056,RP,1,33,24,"4,15",5.72,44.0,65.0,...,0,2,None,CCPG1056,SISTEMAS OPERATIVOS,2020,1S,None,Computación,1


In [446]:
df_repitentes.groupby(["anio", "termino"]).size()

anio  termino
2020  1S          2
      2S          1
2021  1S          3
      2S          2
2022  2S          1
2023  2S          3
2024  1S          1
      2S          6
2025  1S          8
      2S         19
dtype: int64

In [447]:
df_repitentes["ESTADO_MAT_TOMADA_MO"].value_counts()

ESTADO_MAT_TOMADA_MO
RP    44
PF     2
Name: count, dtype: int64

#### 2. Nuevos: Estudiantes que cumplen prerrequisitos y nunca cursaron la materia objetivo

In [448]:
# 1. Obtener las materias prerrequisito por carrera
prereq_por_carrera = (
    df_pre_requisito.groupby('CARRERA')['CODIGOMATERIA']
    .apply(lambda x: set(x.str.strip()))
    .reset_index()
    .rename(columns={'CODIGOMATERIA': 'materias_requeridas'})
)

In [449]:
# 2. Filtrar registros de df_complete donde el estudiante APROBÓ los prerrequisitos
df_complete_prereq_aprobados = df_complete[
    (df_complete['COD_MATERIA_ACAD_MO'].isin(lis_cod_materias_pre)) &
    (df_complete['CARRERA'].isin(lis_carreras_pre)) &
    (df_complete['ESTADO_MAT_TOMADA_MO'] == 'AP')  # Solo materias aprobadas
].copy()

In [450]:
# 3. Agrupar por estudiante y carrera para obtener materias aprobadas
materias_aprobadas_por_estudiante = (
    df_complete_prereq_aprobados.groupby(['COD_ESTUDIANTE', 'CARRERA'])['COD_MATERIA_ACAD_MO']
    .apply(lambda x: set(x.str.strip() if hasattr(x, 'str') else x))
    .reset_index()
    .rename(columns={'COD_MATERIA_ACAD_MO': 'materias_aprobadas'})
)

In [451]:
materias_aprobadas_por_estudiante


,COD_ESTUDIANTE,CARRERA,materias_aprobadas
0,200107944,Computación,{CCPG1049}
1,200123669,Computación,{CCPG1049}
2,200223949,Computación,{CCPG1049}
3,200510428,Computación,{CCPG1049}
4,200607075,Computación,{CCPG1049}
...,...,...,...
754,202408472,Computación,{CCPG1049}
755,202408845,Computación,{CCPG1049}
756,202409512,Computación,{CCPG1049}
757,202409645,Computación,{CCPG1049}


In [452]:
# 4. Hacer merge para comparar con prerrequisitos requeridos
df_comparacion = pd.merge(
    materias_aprobadas_por_estudiante,
    prereq_por_carrera,
    on='CARRERA',
    how='inner'
)

In [453]:
# 5. Verificar que tengan TODOS los prerrequisitos aprobados
df_comparacion['tiene_todos_prereq'] = df_comparacion.apply(
    lambda row: row['materias_requeridas'].issubset(row['materias_aprobadas']),
    axis=1
)

estudiantes_con_prereq_completos = df_comparacion[df_comparacion['tiene_todos_prereq']]

In [454]:
# 6. Filtrar estudiantes que NUNCA han visto la materia objetivo
estudiantes_vieron_objetivo = set(
    df_complete[df_complete['COD_MATERIA_ACAD_MO'] == cod_materia_objetivo]['COD_ESTUDIANTE']
)

In [455]:
# 7. Resultado final: estudiantes elegibles (con prereq aprobados y sin haber visto objetivo)
df_nuevos = estudiantes_con_prereq_completos[
    ~estudiantes_con_prereq_completos['COD_ESTUDIANTE'].isin(estudiantes_vieron_objetivo)
].copy()

In [456]:
print(f"📊 Resumen:")
print(f"  - Estudiantes con todos los prerrequisitos APROBADOS: {len(estudiantes_con_prereq_completos)}")
print(f"  - Estudiantes que vieron {cod_materia_objetivo}: {len(estudiantes_vieron_objetivo)}")
print(f"  - Estudiantes NUEVOS elegibles: {len(df_nuevos)}")
print(f"\nDistribución por carrera:")
print(df_nuevos['CARRERA'].value_counts())

# Ver resultado
df_nuevos[['COD_ESTUDIANTE', 'CARRERA', 'materias_aprobadas', 'materias_requeridas']]

📊 Resumen:
  - Estudiantes con todos los prerrequisitos APROBADOS: 759
  - Estudiantes que vieron CCPG1056: 701
  - Estudiantes NUEVOS elegibles: 181

Distribución por carrera:
CARRERA
Computación    181
Name: count, dtype: int64


,COD_ESTUDIANTE,CARRERA,materias_aprobadas,materias_requeridas
9,201126806,Computación,{CCPG1049},{CCPG1049}
31,201407592,Computación,{CCPG1049},{CCPG1049}
37,201411850,Computación,{CCPG1049},{CCPG1049}
39,201412937,Computación,{CCPG1049},{CCPG1049}
71,201601309,Computación,{CCPG1049},{CCPG1049}
...,...,...,...,...
754,202408472,Computación,{CCPG1049},{CCPG1049}
755,202408845,Computación,{CCPG1049},{CCPG1049}
756,202409512,Computación,{CCPG1049},{CCPG1049}
757,202409645,Computación,{CCPG1049},{CCPG1049}


In [457]:
# 1. Validación final: Verificar que TODOS los prerrequisitos estén cumplidos
print("🔍 Validación de prerrequisitos por estudiante:")
for idx, row in df_nuevos.iterrows():
    faltantes = row['materias_requeridas'] - row['materias_aprobadas']
    if len(faltantes) > 0:
        print(f"  ⚠️ Estudiante {row['COD_ESTUDIANTE']} - Carrera: {row['CARRERA']} - Faltantes: {faltantes}")

# Verificar que no haya faltantes
df_nuevos['prereq_completos'] = df_nuevos.apply(
    lambda row: len(row['materias_requeridas'] - row['materias_aprobadas']) == 0,
    axis=1
)
print(f"\n✅ Todos cumplen prerrequisitos: {df_nuevos['prereq_completos'].all()}")
print(f"Total estudiantes con prerrequisitos completos: {df_nuevos['prereq_completos'].sum()}")

🔍 Validación de prerrequisitos por estudiante:

✅ Todos cumplen prerrequisitos: True
Total estudiantes con prerrequisitos completos: 181


In [458]:
df_nuevos

,COD_ESTUDIANTE,CARRERA,materias_aprobadas,materias_requeridas,tiene_todos_prereq,prereq_completos
9,201126806,Computación,{CCPG1049},{CCPG1049},True,True
31,201407592,Computación,{CCPG1049},{CCPG1049},True,True
37,201411850,Computación,{CCPG1049},{CCPG1049},True,True
39,201412937,Computación,{CCPG1049},{CCPG1049},True,True
71,201601309,Computación,{CCPG1049},{CCPG1049},True,True
...,...,...,...,...,...,...
754,202408472,Computación,{CCPG1049},{CCPG1049},True,True
755,202408845,Computación,{CCPG1049},{CCPG1049},True,True
756,202409512,Computación,{CCPG1049},{CCPG1049},True,True
757,202409645,Computación,{CCPG1049},{CCPG1049},True,True


In [459]:
# cod_materia_objetivo

In [460]:
# para comprobar que ese estudiante no ha visto la materia objetivo y si ha visto las materias requestidas
# df_complete[(df_complete['COD_ESTUDIANTE'] == "201229676") & (df_complete['COD_MATERIA_ACAD_MO'] == "MATG1057")]

#### 3. Unión: DataFrame final con Repitentes y Nuevos

In [461]:
# Agregar columna identificadora del tipo de estudiante
df_repitentes['TIPO_ESTUDIANTE'] = 'REPITENTE'
df_nuevos['TIPO_ESTUDIANTE'] = 'NUEVO'

# Unir ambos dataframes
df_final_nuevos_y_repitentes = pd.concat([df_repitentes, df_nuevos], ignore_index=True)

# Ordenar por tipo y código de estudiante
df_final_nuevos_y_repitentes = df_final_nuevos_y_repitentes.sort_values(['TIPO_ESTUDIANTE', 'COD_ESTUDIANTE']).reset_index(drop=True)

print(f"\n📊 RESUMEN FINAL:")
print(f"  • Repitentes: {df_repitentes.shape[0]}")
print(f"  • Nuevos: {df_nuevos.shape[0]}")
print(f"  • TOTAL: {df_final_nuevos_y_repitentes.shape[0]}")
print(f"\nDistribución por tipo:")
print(df_final_nuevos_y_repitentes['TIPO_ESTUDIANTE'].value_counts())

df_final_nuevos_y_repitentes


📊 RESUMEN FINAL:
  • Repitentes: 46
  • Nuevos: 181
  • TOTAL: 227

Distribución por tipo:
TIPO_ESTUDIANTE
NUEVO        181
REPITENTE     46
Name: count, dtype: int64


,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num,TIPO_ESTUDIANTE,materias_aprobadas,materias_requeridas,tiene_todos_prereq,prereq_completos
0,201126806,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Computación,NaN,NUEVO,{CCPG1049},{CCPG1049},True,True
1,201407592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Computación,NaN,NUEVO,{CCPG1049},{CCPG1049},True,True
2,201411850,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Computación,NaN,NUEVO,{CCPG1049},{CCPG1049},True,True
3,201412937,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Computación,NaN,NUEVO,{CCPG1049},{CCPG1049},True,True
4,201601309,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Computación,NaN,NUEVO,{CCPG1049},{CCPG1049},True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222,202207890,CCPG1056,RP,1,0,0,"0,00",5.72,23.0,66.0,...,2025,2S,"7,72",Computación,2.0,REPITENTE,NaN,NaN,NaN,NaN
223,202211470,CCPG1056,RP,1,0,0,"0,00",5.72,29.0,66.0,...,2025,2S,"7,35",Computación,2.0,REPITENTE,NaN,NaN,NaN,NaN
224,202212817,CCPG1056,RP,1,0,0,"0,00",5.72,27.0,66.0,...,2025,2S,"7,53",Computación,2.0,REPITENTE,NaN,NaN,NaN,NaN
225,202212965,CCPG1056,RP,1,42,16,"2,90",5.72,28.0,79.0,...,2025,1S,None,Computación,1.0,REPITENTE,NaN,NaN,NaN,NaN


In [462]:
# Verificación: mostrar ejemplos de cada tipo
print("\n🔍 Ejemplos de REPITENTES:")
print(df_final_nuevos_y_repitentes[df_final_nuevos_y_repitentes['TIPO_ESTUDIANTE'] == 'REPITENTE'][['COD_ESTUDIANTE', 'TIPO_ESTUDIANTE', 'ESTADO_MAT_TOMADA_MO', 'anio', 'termino']].head())

print("\n🔍 Ejemplos de NUEVOS:")
print(df_final_nuevos_y_repitentes[df_final_nuevos_y_repitentes['TIPO_ESTUDIANTE'] == 'NUEVO'][['COD_ESTUDIANTE', 'TIPO_ESTUDIANTE']].head())


🔍 Ejemplos de REPITENTES:
    COD_ESTUDIANTE TIPO_ESTUDIANTE ESTADO_MAT_TOMADA_MO  anio termino
181      200123669       REPITENTE                   RP  2021      2S
182      200525939       REPITENTE                   RP  2021      1S
183      200533511       REPITENTE                   RP  2024      2S
184      201020203       REPITENTE                   RP  2021      2S
185      201122213       REPITENTE                   RP  2020      1S

🔍 Ejemplos de NUEVOS:
  COD_ESTUDIANTE TIPO_ESTUDIANTE
0      201126806           NUEVO
1      201407592           NUEVO
2      201411850           NUEVO
3      201412937           NUEVO
4      201601309           NUEVO


In [463]:
df_final_nuevos_y_repitentes

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num,TIPO_ESTUDIANTE,materias_aprobadas,materias_requeridas,tiene_todos_prereq,prereq_completos
0,201126806,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Computación,NaN,NUEVO,{CCPG1049},{CCPG1049},True,True
1,201407592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Computación,NaN,NUEVO,{CCPG1049},{CCPG1049},True,True
2,201411850,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Computación,NaN,NUEVO,{CCPG1049},{CCPG1049},True,True
3,201412937,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Computación,NaN,NUEVO,{CCPG1049},{CCPG1049},True,True
4,201601309,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Computación,NaN,NUEVO,{CCPG1049},{CCPG1049},True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222,202207890,CCPG1056,RP,1,0,0,"0,00",5.72,23.0,66.0,...,2025,2S,"7,72",Computación,2.0,REPITENTE,NaN,NaN,NaN,NaN
223,202211470,CCPG1056,RP,1,0,0,"0,00",5.72,29.0,66.0,...,2025,2S,"7,35",Computación,2.0,REPITENTE,NaN,NaN,NaN,NaN
224,202212817,CCPG1056,RP,1,0,0,"0,00",5.72,27.0,66.0,...,2025,2S,"7,53",Computación,2.0,REPITENTE,NaN,NaN,NaN,NaN
225,202212965,CCPG1056,RP,1,42,16,"2,90",5.72,28.0,79.0,...,2025,1S,None,Computación,1.0,REPITENTE,NaN,NaN,NaN,NaN


In [464]:
df_final_nuevos_y_repitentes.keys() # mejor solo agregar la dificultad de la materia objetivo

Index(['COD_ESTUDIANTE', 'COD_MATERIA_ACAD_MO', 'ESTADO_MAT_TOMADA_MO',
       'VEZ_TOMADA_MO', 'NOTA1_MO', 'NOTA2MO', 'PROMEDIO_MO', 'DIFICULTAD_MO',
       'T_MAT_TOMADAS', 'PROM_1PARCIAL', 'PROM_2PARCIAL',
       'PROM_CALIFICACIONES', 'MAT_APROBADAS', 'PROM_CALIF_APROBADAS',
       'TERMINOS_REGISTRADOS', 'PERDIO_CARRERA', 'PROM_MAT_REPROBADAS1',
       'PROM_MAT_REPROBADAS2', 'PROM_MAT_REPROBADAS3', 'MUY_FACIL', 'FACIL',
       'MODERADA', 'DIFICIL', 'MUY_DIFICIL', 'promedio_general',
       'CODIGOMATERIA', 'MATERIA', 'anio', 'termino', 'PROMEDIO_GENERAL',
       'CARRERA', 'termino_num', 'TIPO_ESTUDIANTE', 'materias_aprobadas',
       'materias_requeridas', 'tiene_todos_prereq', 'prereq_completos'],
      dtype='object')

#### Obtener los ultimos registros de los estudiantes que veran la materia objetivo (df_final_nuevos_y_repitentes["COD_ESTUDIANTE"].unique())

In [465]:
df_final_nuevos_y_repitentes["COD_ESTUDIANTE"].nunique(), df_final_nuevos_y_repitentes.shape[0]

(227, 227)

In [466]:
# 1. Filtrar el df_complete para tener solo los estudiantes de df_final_nuevos_y_repitentes
estudiantes_ids = df_final_nuevos_y_repitentes["COD_ESTUDIANTE"].unique()
df_filtrado = df_complete[df_complete["COD_ESTUDIANTE"].isin(estudiantes_ids)].copy()
# eliminar los que tienen una VEZ_TOMADA_MO = 3 y ESTADO_MAT_TOMADA_MO = RP o PF
df_filtrado = df_filtrado[~((df_filtrado["VEZ_TOMADA_MO"] == 3) & (df_filtrado["ESTADO_MAT_TOMADA_MO"].isin(["RP", "PF"])))]

In [467]:
# 2. Crear un identificador numérico único para el periodo (Año + Término)
# Multiplicamos el año por 100 para que 2020-2 sea mayor que 2020-1 de forma matemática
df_filtrado['periodo_id'] = (df_filtrado['anio'].astype(int) * 100) + df_filtrado['termino_num']

In [468]:
# 3. Calcular el periodo MÁXIMO por cada estudiante y asignarlo a una nueva columna
# transform('max') repite el valor máximo del grupo en todas las filas de ese estudiante
df_filtrado['max_periodo'] = df_filtrado.groupby('COD_ESTUDIANTE')['periodo_id'].transform('max')

In [469]:
# 4. Filtrar las filas donde el periodo actual coincide con el máximo encontrado
df_resultado = df_filtrado[df_filtrado['periodo_id'] == df_filtrado['max_periodo']].copy()

In [470]:
# Limpiar columnas auxiliares
df_resultado.drop(columns=['periodo_id', 'max_periodo'], inplace=True)

In [471]:
df_resultado

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num
8011,201304792,CCPG1044,RP,1,66,52,"5,90",6.14,64.0,52.0,...,0,2,NaN,CCPG1044,INTELIGENCIA ARTIFICIAL,2020,1S,NaN,Computación,1
8976,201304792,CCPG1056,RP,1,21,6,"1,35",5.72,64.0,52.0,...,0,2,NaN,CCPG1056,SISTEMAS OPERATIVOS,2020,1S,NaN,Computación,1
44479,201122213,CCPG1056,RP,2,46,21,"3,50",5.72,47.0,64.0,...,0,1,NaN,CCPG1056,SISTEMAS OPERATIVOS,2020,2S,NaN,Computación,2
78073,200525939,CCPG1056,RP,1,22,2,"1,20",5.72,18.0,64.0,...,0,1,NaN,CCPG1056,SISTEMAS OPERATIVOS,2021,1S,NaN,Computación,1
109112,201901477,CCPG1034,RP,2,49,45,"4,58",6.06,24.0,66.0,...,0,6,NaN,CCPG1034,ESTRUCTURAS DE DATOS,2021,2S,NaN,Computación,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366334,202211017,TLMG1032,AC,1,0,0,"0,00",7.19,23.0,90.0,...,1,3,NaN,TLMG1032,REDES DE DATOS,2025,2S,"9,06",Computación,2
366335,202315248,TLMG1032,AC,1,0,0,"0,00",7.19,16.0,78.0,...,1,3,NaN,TLMG1032,REDES DE DATOS,2025,2S,"8,29",Computación,2
366339,202313078,TLMG1032,AC,1,0,0,"0,00",7.19,18.0,75.0,...,1,5,NaN,TLMG1032,REDES DE DATOS,2025,2S,"7,54",Computación,2
366354,201705316,TLMG1034,AC,2,0,0,"0,00",6.88,80.0,71.0,...,1,1,NaN,TLMG1034,SISTEMAS EN LA NUBE,2025,2S,"7,84",Telemática,2


In [472]:
list_student_resultado = df_resultado["COD_ESTUDIANTE"].unique()

In [473]:
len(list_student_resultado), len(estudiantes_ids)

(227, 227)

In [474]:
indice_busq_para_comprobar = 5

In [475]:
df_resultado[df_resultado["COD_ESTUDIANTE"] == list_student_resultado[indice_busq_para_comprobar]]

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num
116272,201411850,ESTG1034,AP,3,75,53,"7,18",5.84,47.0,52.0,...,0,2,NaN,ESTG1034,ESTADÍSTICA,2021,2S,NaN,Computación,2


In [476]:
df_complete[df_complete["COD_ESTUDIANTE"] == list_student_resultado[indice_busq_para_comprobar]].sort_values(by=['anio', 'termino_num'], ascending=[False, False])

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num
116272,201411850,ESTG1034,AP,3,75,53,"7,18",5.84,47.0,52.0,...,0,2,NaN,ESTG1034,ESTADÍSTICA,2021,2S,NaN,Computación,2
126099,201411850,MATG1051,RP,3,37,29,"4,66",4.90,47.0,52.0,...,0,2,NaN,MATG1051,MATEMÁTICAS DISCRETAS,2021,2S,NaN,Computación,2
83462,201411850,ESTG1034,RP,2,16,3,"1,17",5.84,44.0,53.0,...,0,3,NaN,ESTG1034,ESTADÍSTICA,2021,1S,NaN,Computación,1
94221,201411850,MATG1051,RP,2,41,0,"1,79",4.90,44.0,53.0,...,0,3,NaN,MATG1051,MATEMÁTICAS DISCRETAS,2021,1S,NaN,Computación,1
101056,201411850,SOFG1009,RP,1,56,5,"3,05",7.06,44.0,53.0,...,0,3,NaN,SOFG1009,LENGUAJES DE PROGRAMACIÓN,2021,1S,NaN,Computación,1
42655,201411850,CCPG1042,AP,1,73,43,"6,73",6.78,39.0,52.0,...,0,4,NaN,CCPG1042,DISEÑO DE SOFTWARE,2020,2S,NaN,Computación,2
43759,201411850,CCPG1046,AP,1,80,83,"8,10",7.10,39.0,52.0,...,0,4,NaN,CCPG1046,INTERACCIÓN HUMANO COMPUTADOR,2020,2S,NaN,Computación,2
43892,201411850,CCPG1049,AP,1,52,93,"8,07",7.00,39.0,52.0,...,0,4,NaN,CCPG1049,ORGANIZACIÓN DE COMPUTADORES,2020,2S,NaN,Computación,2
49753,201411850,ESTG1034,RP,1,57,39,"4,65",5.84,39.0,52.0,...,0,4,NaN,ESTG1034,ESTADÍSTICA,2020,2S,NaN,Computación,2
6912,201411850,CCPG1034,AP,1,68,40,"6,63",6.06,34.0,49.0,...,0,5,NaN,CCPG1034,ESTRUCTURAS DE DATOS,2020,1S,NaN,Computación,1


In [477]:
df_resultado

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num
8011,201304792,CCPG1044,RP,1,66,52,"5,90",6.14,64.0,52.0,...,0,2,NaN,CCPG1044,INTELIGENCIA ARTIFICIAL,2020,1S,NaN,Computación,1
8976,201304792,CCPG1056,RP,1,21,6,"1,35",5.72,64.0,52.0,...,0,2,NaN,CCPG1056,SISTEMAS OPERATIVOS,2020,1S,NaN,Computación,1
44479,201122213,CCPG1056,RP,2,46,21,"3,50",5.72,47.0,64.0,...,0,1,NaN,CCPG1056,SISTEMAS OPERATIVOS,2020,2S,NaN,Computación,2
78073,200525939,CCPG1056,RP,1,22,2,"1,20",5.72,18.0,64.0,...,0,1,NaN,CCPG1056,SISTEMAS OPERATIVOS,2021,1S,NaN,Computación,1
109112,201901477,CCPG1034,RP,2,49,45,"4,58",6.06,24.0,66.0,...,0,6,NaN,CCPG1034,ESTRUCTURAS DE DATOS,2021,2S,NaN,Computación,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
366334,202211017,TLMG1032,AC,1,0,0,"0,00",7.19,23.0,90.0,...,1,3,NaN,TLMG1032,REDES DE DATOS,2025,2S,"9,06",Computación,2
366335,202315248,TLMG1032,AC,1,0,0,"0,00",7.19,16.0,78.0,...,1,3,NaN,TLMG1032,REDES DE DATOS,2025,2S,"8,29",Computación,2
366339,202313078,TLMG1032,AC,1,0,0,"0,00",7.19,18.0,75.0,...,1,5,NaN,TLMG1032,REDES DE DATOS,2025,2S,"7,54",Computación,2
366354,201705316,TLMG1034,AC,2,0,0,"0,00",6.88,80.0,71.0,...,1,1,NaN,TLMG1034,SISTEMAS EN LA NUBE,2025,2S,"7,84",Telemática,2


#### Quedarme con el mayor y actualiza su DIFICULTAD_MO

In [478]:
dificultad_materia_obj = df_complete[df_complete["COD_MATERIA_ACAD_MO"] == cod_materia_objetivo]["DIFICULTAD_MO"].unique()[0]

In [479]:
# 1. Calcular la diferencia absoluta con la dificultad objetivo
# (Asumiendo que 'dificultad_materia_obj' es una variable con el valor numérico)
df_resultado['diff_gap'] = (df_resultado['DIFICULTAD_MO'].astype(float) - dificultad_materia_obj).abs()

In [480]:
# 2. Ordenar los datos
# - Primero por estudiante (para agrupar)
# - Segundo por 'diff_gap' ASCENDENTE (el más cercano a 0 es el más similar)
# - Tercero por 'PROMEDIO_MO' DESCENDENTE (el más alto gana en caso de empate o cercanía similar)
df_ordenado = df_resultado.sort_values(
    by=['COD_ESTUDIANTE', 'diff_gap', 'PROMEDIO_MO'],
    ascending=[True, True, False]
)

In [481]:
# agregar la columna CANT_ACTUAL_MAT_TOMADAS  a df_ordenado antes de eliminar los duplicados
# df_ordenado['CANT_ACTUAL_MAT_TOMADAS'] = df_ordenado.groupby('COD_ESTUDIANTE').cumcount() + 1
df_ordenado['CANT_ACTUAL_MAT_TOMADAS'] = df_ordenado.groupby('COD_ESTUDIANTE')['COD_MATERIA_ACAD_MO'].transform('count')

In [482]:
# 3. Quedarse con la primera fila de cada estudiante (la mejor opción según el orden)
df_seleccion_final = df_ordenado.drop_duplicates(subset='COD_ESTUDIANTE', keep='first').copy()

In [483]:
# Limpieza opcional
df_seleccion_final.drop(columns=['diff_gap'], inplace=True)

In [484]:
df_seleccion_final

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num,CANT_ACTUAL_MAT_TOMADAS
142043,200123669,CCPG1056,RP,2,47,49,"4,80",5.72,15.0,58.0,...,3,NaN,CCPG1056,SISTEMAS OPERATIVOS,2022,1S,NaN,Computación,1,3
78073,200525939,CCPG1056,RP,1,22,2,"1,20",5.72,18.0,64.0,...,1,NaN,CCPG1056,SISTEMAS OPERATIVOS,2021,1S,NaN,Computación,1,1
342340,200533511,CCPG1044,AC,2,0,0,"0,00",6.14,17.0,50.0,...,1,NaN,CCPG1044,INTELIGENCIA ARTIFICIAL,2025,2S,NaN,Computación,2,1
227794,201020203,CCPG1053,PF,1,7,0,"0,84",8.47,57.0,56.0,...,0,NaN,CCPG1053,SEGURIDAD DE LA INFORMACIÓN,2023,2S,NaN,Computación,2,1
44479,201122213,CCPG1056,RP,2,46,21,"3,50",5.72,47.0,64.0,...,1,NaN,CCPG1056,SISTEMAS OPERATIVOS,2020,2S,NaN,Computación,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
340879,202408472,CCPG1034,AC,1,0,0,"0,00",6.06,13.0,78.0,...,5,NaN,CCPG1034,ESTRUCTURAS DE DATOS,2025,2S,"8,46",Computación,2,5
340846,202408845,CCPG1034,AC,1,0,0,"0,00",6.06,15.0,82.0,...,5,NaN,CCPG1034,ESTRUCTURAS DE DATOS,2025,2S,"8,24",Computación,2,6
342682,202409512,CCPG1051,AC,1,0,0,"0,00",6.42,16.0,77.0,...,4,NaN,CCPG1051,PROGRAMACIÓN DE SISTEMAS,2025,2S,"8,05",Computación,2,5
347909,202409645,ESTG1034,AC,1,0,0,"0,00",5.84,14.0,87.0,...,5,NaN,ESTG1034,ESTADÍSTICA,2025,2S,"8,80",Computación,2,5


In [485]:
df_seleccion_final["COD_ESTUDIANTE"].nunique(), df_final_nuevos_y_repitentes["COD_ESTUDIANTE"].nunique(), df_resultado["COD_ESTUDIANTE"].nunique()

(227, 227, 227)

In [486]:
df_seleccion_final["DIFICULTAD_MO"] = dificultad_materia_obj
df_seleccion_final["COD_MATERIA_ACAD_MO"] = cod_materia_objetivo
df_seleccion_final["ESTADO_MAT_TOMADA_MO"] = "AC"

In [487]:
df_seleccion_final

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num,CANT_ACTUAL_MAT_TOMADAS
142043,200123669,CCPG1056,AC,2,47,49,"4,80",5.72,15.0,58.0,...,3,NaN,CCPG1056,SISTEMAS OPERATIVOS,2022,1S,NaN,Computación,1,3
78073,200525939,CCPG1056,AC,1,22,2,"1,20",5.72,18.0,64.0,...,1,NaN,CCPG1056,SISTEMAS OPERATIVOS,2021,1S,NaN,Computación,1,1
342340,200533511,CCPG1056,AC,2,0,0,"0,00",5.72,17.0,50.0,...,1,NaN,CCPG1044,INTELIGENCIA ARTIFICIAL,2025,2S,NaN,Computación,2,1
227794,201020203,CCPG1056,AC,1,7,0,"0,84",5.72,57.0,56.0,...,0,NaN,CCPG1053,SEGURIDAD DE LA INFORMACIÓN,2023,2S,NaN,Computación,2,1
44479,201122213,CCPG1056,AC,2,46,21,"3,50",5.72,47.0,64.0,...,1,NaN,CCPG1056,SISTEMAS OPERATIVOS,2020,2S,NaN,Computación,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
340879,202408472,CCPG1056,AC,1,0,0,"0,00",5.72,13.0,78.0,...,5,NaN,CCPG1034,ESTRUCTURAS DE DATOS,2025,2S,"8,46",Computación,2,5
340846,202408845,CCPG1056,AC,1,0,0,"0,00",5.72,15.0,82.0,...,5,NaN,CCPG1034,ESTRUCTURAS DE DATOS,2025,2S,"8,24",Computación,2,6
342682,202409512,CCPG1056,AC,1,0,0,"0,00",5.72,16.0,77.0,...,4,NaN,CCPG1051,PROGRAMACIÓN DE SISTEMAS,2025,2S,"8,05",Computación,2,5
347909,202409645,CCPG1056,AC,1,0,0,"0,00",5.72,14.0,87.0,...,5,NaN,ESTG1034,ESTADÍSTICA,2025,2S,"8,80",Computación,2,5


### Usar el modelo para 2026-1S

In [488]:
df_seleccion_final["anio"] = df_seleccion_final["anio"].astype(int)

In [489]:
df_seleccion_final

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num,CANT_ACTUAL_MAT_TOMADAS
142043,200123669,CCPG1056,AC,2,47,49,"4,80",5.72,15.0,58.0,...,3,NaN,CCPG1056,SISTEMAS OPERATIVOS,2022,1S,NaN,Computación,1,3
78073,200525939,CCPG1056,AC,1,22,2,"1,20",5.72,18.0,64.0,...,1,NaN,CCPG1056,SISTEMAS OPERATIVOS,2021,1S,NaN,Computación,1,1
342340,200533511,CCPG1056,AC,2,0,0,"0,00",5.72,17.0,50.0,...,1,NaN,CCPG1044,INTELIGENCIA ARTIFICIAL,2025,2S,NaN,Computación,2,1
227794,201020203,CCPG1056,AC,1,7,0,"0,84",5.72,57.0,56.0,...,0,NaN,CCPG1053,SEGURIDAD DE LA INFORMACIÓN,2023,2S,NaN,Computación,2,1
44479,201122213,CCPG1056,AC,2,46,21,"3,50",5.72,47.0,64.0,...,1,NaN,CCPG1056,SISTEMAS OPERATIVOS,2020,2S,NaN,Computación,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
340879,202408472,CCPG1056,AC,1,0,0,"0,00",5.72,13.0,78.0,...,5,NaN,CCPG1034,ESTRUCTURAS DE DATOS,2025,2S,"8,46",Computación,2,5
340846,202408845,CCPG1056,AC,1,0,0,"0,00",5.72,15.0,82.0,...,5,NaN,CCPG1034,ESTRUCTURAS DE DATOS,2025,2S,"8,24",Computación,2,6
342682,202409512,CCPG1056,AC,1,0,0,"0,00",5.72,16.0,77.0,...,4,NaN,CCPG1051,PROGRAMACIÓN DE SISTEMAS,2025,2S,"8,05",Computación,2,5
347909,202409645,CCPG1056,AC,1,0,0,"0,00",5.72,14.0,87.0,...,5,NaN,ESTG1034,ESTADÍSTICA,2025,2S,"8,80",Computación,2,5


In [490]:
df_final_nuevos_y_repitentes

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num,TIPO_ESTUDIANTE,materias_aprobadas,materias_requeridas,tiene_todos_prereq,prereq_completos
0,201126806,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Computación,NaN,NUEVO,{CCPG1049},{CCPG1049},True,True
1,201407592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Computación,NaN,NUEVO,{CCPG1049},{CCPG1049},True,True
2,201411850,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Computación,NaN,NUEVO,{CCPG1049},{CCPG1049},True,True
3,201412937,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Computación,NaN,NUEVO,{CCPG1049},{CCPG1049},True,True
4,201601309,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,Computación,NaN,NUEVO,{CCPG1049},{CCPG1049},True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
222,202207890,CCPG1056,RP,1,0,0,"0,00",5.72,23.0,66.0,...,2025,2S,"7,72",Computación,2.0,REPITENTE,NaN,NaN,NaN,NaN
223,202211470,CCPG1056,RP,1,0,0,"0,00",5.72,29.0,66.0,...,2025,2S,"7,35",Computación,2.0,REPITENTE,NaN,NaN,NaN,NaN
224,202212817,CCPG1056,RP,1,0,0,"0,00",5.72,27.0,66.0,...,2025,2S,"7,53",Computación,2.0,REPITENTE,NaN,NaN,NaN,NaN
225,202212965,CCPG1056,RP,1,42,16,"2,90",5.72,28.0,79.0,...,2025,1S,None,Computación,1.0,REPITENTE,NaN,NaN,NaN,NaN


In [491]:
df_final_nuevos_y_repitentes[df_final_nuevos_y_repitentes["TIPO_ESTUDIANTE"] == "NUEVO"]["COD_ESTUDIANTE"].nunique()

181

In [492]:
# **************************************************************************
# # Crear un set una sola vez (fuera del apply)
# estudiantes_nuevos = set(
#     df_final_nuevos_y_repitentes[
#         df_final_nuevos_y_repitentes["TIPO_ESTUDIANTE"] == "NUEVO"
#     ]["COD_ESTUDIANTE"].unique()
# )

# # Aplicar la lógica de forma más eficiente
# df_seleccion_final["VEZ_TOMADA_MO"] = df_seleccion_final.apply(
#     lambda row: 1 if row['COD_ESTUDIANTE'] in estudiantes_nuevos else row['VEZ_TOMADA_MO'] + 1,
#     axis=1
# )

In [493]:
df_seleccion_final

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA,anio,termino,PROMEDIO_GENERAL,CARRERA,termino_num,CANT_ACTUAL_MAT_TOMADAS
142043,200123669,CCPG1056,AC,2,47,49,"4,80",5.72,15.0,58.0,...,3,NaN,CCPG1056,SISTEMAS OPERATIVOS,2022,1S,NaN,Computación,1,3
78073,200525939,CCPG1056,AC,1,22,2,"1,20",5.72,18.0,64.0,...,1,NaN,CCPG1056,SISTEMAS OPERATIVOS,2021,1S,NaN,Computación,1,1
342340,200533511,CCPG1056,AC,2,0,0,"0,00",5.72,17.0,50.0,...,1,NaN,CCPG1044,INTELIGENCIA ARTIFICIAL,2025,2S,NaN,Computación,2,1
227794,201020203,CCPG1056,AC,1,7,0,"0,84",5.72,57.0,56.0,...,0,NaN,CCPG1053,SEGURIDAD DE LA INFORMACIÓN,2023,2S,NaN,Computación,2,1
44479,201122213,CCPG1056,AC,2,46,21,"3,50",5.72,47.0,64.0,...,1,NaN,CCPG1056,SISTEMAS OPERATIVOS,2020,2S,NaN,Computación,2,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
340879,202408472,CCPG1056,AC,1,0,0,"0,00",5.72,13.0,78.0,...,5,NaN,CCPG1034,ESTRUCTURAS DE DATOS,2025,2S,"8,46",Computación,2,5
340846,202408845,CCPG1056,AC,1,0,0,"0,00",5.72,15.0,82.0,...,5,NaN,CCPG1034,ESTRUCTURAS DE DATOS,2025,2S,"8,24",Computación,2,6
342682,202409512,CCPG1056,AC,1,0,0,"0,00",5.72,16.0,77.0,...,4,NaN,CCPG1051,PROGRAMACIÓN DE SISTEMAS,2025,2S,"8,05",Computación,2,5
347909,202409645,CCPG1056,AC,1,0,0,"0,00",5.72,14.0,87.0,...,5,NaN,ESTG1034,ESTADÍSTICA,2025,2S,"8,80",Computación,2,5


In [494]:
# Agregar el GPA general al dataframe final
df_seleccion_final = pd.merge(
    df_seleccion_final,
    df_gpa_general[['COD_ESTUDIANTE', 'ANIO', 'TERMINO', 'NUMERADOR', 'DENOMINADOR']],
    left_on=['COD_ESTUDIANTE', 'anio', "termino"],
    right_on=['COD_ESTUDIANTE', 'ANIO', "TERMINO"],
    how='left'
)

# Calcular GPA
df_seleccion_final['GPA'] = round(df_seleccion_final['NUMERADOR'] / df_seleccion_final['DENOMINADOR'], 2)
# Limpiar columnas auxiliares si es necesario
df_seleccion_final = df_seleccion_final.drop(columns=['NUMERADOR', 'DENOMINADOR', 'ANIO'], errors='ignore')

In [495]:
df_seleccion_final = pd.merge(
    df_seleccion_final,
    df_socioeconomico[['CODESTUDIANTE', 'GASTOS_RUBRO']],
    left_on='COD_ESTUDIANTE',
    right_on='CODESTUDIANTE',
    how='left',
    # validate='one_to_one'
).drop(columns=['CODESTUDIANTE']).copy()

In [496]:
df_seleccion_final["GASTOS_RUBRO"].fillna(df_seleccion_final["GASTOS_RUBRO"].mean(), inplace=True)

C:\Users\saraujo\AppData\Local\Temp\ipykernel_80540\1458026258.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_seleccion_final["GASTOS_RUBRO"].fillna(df_seleccion_final["GASTOS_RUBRO"].mean(), inplace=True)


In [497]:
# Interacciones
df_seleccion_final['VEZ_x_DIFICULTAD'] =  (df_seleccion_final['VEZ_TOMADA_MO'] * df_seleccion_final['DIFICULTAD_MO']) ** 2

# Ratios
df_seleccion_final['RATIO_APROBADAS'] = np.log(df_seleccion_final['MAT_APROBADAS'] / (df_seleccion_final['T_MAT_TOMADAS'] + 1))
df_seleccion_final['TASA_REPROBACION'] = df_seleccion_final['PROM_MAT_REPROBADAS1'] / (df_seleccion_final['T_MAT_TOMADAS'] + 1)
# No lineales
df_seleccion_final['GPA_CUADRADO'] = df_seleccion_final['GPA'] ** 2
df_seleccion_final['LOG_CANT_MAT'] = np.log1p(df_seleccion_final['CANT_ACTUAL_MAT_TOMADAS'])
df_seleccion_final['LOG_GASTOS_RUBRO'] = np.log1p(df_seleccion_final['GASTOS_RUBRO'])

In [498]:
df_socioeconomico["FECHANACIMIENTO"] = pd.to_datetime(df_socioeconomico["FECHANACIMIENTO"], errors='coerce', format="%Y-%m-%d")
anio_ingreso = df_socioeconomico["ANIO_TERMINO_INGRESO"].str.split(' ').str[0].astype(int)
df_socioeconomico["edad_ingreso"] = (anio_ingreso - df_socioeconomico["FECHANACIMIENTO"].dt.year)

In [499]:
# apartir de la columna IDIOMAS se genere otra que sea NUMERO_IDIOMAS y si es nan sea cero
df_socioeconomico["NUMERO_IDIOMAS"] = df_socioeconomico["IDIOMAS"].fillna("0").apply(
    lambda x: 0 if x == "0" or pd.isna(x) or str(x).lower() in ['nan', 'ninguno', 'no', ''] 
    else len(str(x).split(';')) if ';' in str(x) 
    else len(str(x).split(',')) if ',' in str(x)
    else 1 if str(x).strip() != '' 
    else 0
)

In [500]:
df_socioeconomico_interes =  df_socioeconomico[["CODESTUDIANTE", "PORCENTAJEDISCAPACIDAD", "VECESBUSENTRADA", "VECESBUSSALIDA", "NUMERO_IDIOMAS", "CANTIDADCUARTOS", "CANTIDADBANIO", "edad_ingreso", "TIPOCOLEGIO", "BECACOLEGIO", "TIENEDISCAPACIDAD", "TIPODISCAPACIDAD", "ESTADOCIVIL", "OTROSIDIOMAS", "TIEMPOPROMEDIOLLEGARESPOL", "NIVELINGLES", "NIVELINSTRUCCIONPADRE", "NIVELINSTRUCCIONMADRE", "ESTADOCIVILPADRES", "FAMILIARDISCAPACIDAD", "FAMILIARENFERMEDAD", "TIPOPARROQUIA", "VIVEGRUPOFAMILIAR", "SEXO"]]
# todas las columnas str a minusculas
for col in df_socioeconomico_interes.select_dtypes(include=['object']).columns:
    df_socioeconomico_interes[col] = df_socioeconomico_interes[col].str.lower().str.strip()

C:\Users\saraujo\AppData\Local\Temp\ipykernel_80540\2512511134.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_socioeconomico_interes[col] = df_socioeconomico_interes[col].str.lower().str.strip()


In [501]:
df_seleccion_final_2 = pd.merge(
    df_seleccion_final,
    df_socioeconomico_interes,
    left_on='COD_ESTUDIANTE',
    right_on='CODESTUDIANTE',
    how='left',
    # validate='one_to_one'
)

In [502]:
df_seleccion_final_2["TIPODISCAPACIDAD"] = df_seleccion_final_2["TIPODISCAPACIDAD"].fillna("no definida")
df_seleccion_final_2["TIENEDISCAPACIDAD"] = df_seleccion_final_2["TIENEDISCAPACIDAD"].fillna("n")
df_seleccion_final_2[df_seleccion_final_2["TIENEDISCAPACIDAD"] == "nan"] = "n"
df_seleccion_final_2["TIPOCOLEGIO"] = df_seleccion_final_2["TIPOCOLEGIO"].fillna("nacional")
df_seleccion_final_2["BECACOLEGIO"] = df_seleccion_final_2["BECACOLEGIO"].fillna("ninguna")
df_seleccion_final_2["ESTADOCIVIL"] = df_seleccion_final_2["ESTADOCIVIL"].fillna("soltero")
df_seleccion_final_2["OTROSIDIOMAS"] = df_seleccion_final_2["OTROSIDIOMAS"].fillna("no")
df_seleccion_final_2["TIEMPOPROMEDIOLLEGARESPOL"] = df_seleccion_final_2["TIEMPOPROMEDIOLLEGARESPOL"].fillna("61 a 90 minutos")
df_seleccion_final_2["NIVELINGLES"] = df_seleccion_final_2["NIVELINGLES"].fillna("básico")
df_seleccion_final_2["NIVELINSTRUCCIONPADRE"] = df_seleccion_final_2["NIVELINSTRUCCIONPADRE"].fillna("desconocido")
df_seleccion_final_2["NIVELINSTRUCCIONMADRE"] = df_seleccion_final_2["NIVELINSTRUCCIONMADRE"].fillna("desconocido")
df_seleccion_final_2["ESTADOCIVILPADRES"] = df_seleccion_final_2["ESTADOCIVILPADRES"].fillna("unión de hecho")
df_seleccion_final_2["FAMILIARDISCAPACIDAD"] = df_seleccion_final_2["FAMILIARDISCAPACIDAD"].fillna("no")
df_seleccion_final_2["FAMILIARENFERMEDAD"] = df_seleccion_final_2["FAMILIARENFERMEDAD"].fillna("no")
df_seleccion_final_2["TIPOPARROQUIA"] = df_seleccion_final_2["TIPOPARROQUIA"].fillna("urbana")
df_seleccion_final_2["VIVEGRUPOFAMILIAR"] = df_seleccion_final_2["VIVEGRUPOFAMILIAR"].fillna("si")
df_seleccion_final_2["SEXO"] = df_seleccion_final_2["SEXO"].fillna("masculino")
df_seleccion_final_2["PERDIO_CARRERA"] = df_seleccion_final_2["PERDIO_CARRERA"].str.lower()


In [503]:
if df_seleccion_final_2[df_seleccion_final_2["FAMILIARDISCAPACIDAD"] == "yo mismo (estudiante):no tiene discapacidad;pareja:no tiene discapacidad;hijo(a):intelectual (retraso mental);hijo(a):no tiene discapacidad;"]["FAMILIARDISCAPACIDAD"].size > 0:
    # entonces reemplazar ese valor por "no"

    df_seleccion_final_2.loc[df_seleccion_final_2["FAMILIARDISCAPACIDAD"] == "yo mismo (estudiante):no tiene discapacidad;pareja:no tiene discapacidad;hijo(a):intelectual (retraso mental);hijo(a):no tiene discapacidad;", "FAMILIARDISCAPACIDAD"]  = "no"

In [504]:
# Diagnosticar qué columna tiene valores desconocidos
print("🔍 Verificando valores en cada columna categórica...\n")

encoded_columns = {
    'TIPOCOLEGIO': 'TIPOCOLEGIO_encoded',
    'BECACOLEGIO': 'BECACOLEGIO_encoded',
    'COD_MATERIA_ACAD_MO': 'COD_MATERIA_ACAD_MO_encoded',
    'TIENEDISCAPACIDAD': 'TIENEDISCAPACIDAD_encoded',
    'TIPODISCAPACIDAD': 'TIPODISCAPACIDAD_encoded',
    'ESTADOCIVIL': 'ESTADOCIVIL_encoded',
    'OTROSIDIOMAS': 'OTROSIDIOMAS_encoded',
    'TIEMPOPROMEDIOLLEGARESPOL': 'TIEMPOPROMEDIOLLEGARESPOL_encoded',
    'NIVELINGLES': 'NIVELINGLES_encoded',
    'NIVELINSTRUCCIONPADRE': 'NIVELINSTRUCCIONPADRE_encoded',
    'NIVELINSTRUCCIONMADRE': 'NIVELINSTRUCCIONMADRE_encoded',
    'ESTADOCIVILPADRES': 'ESTADOCIVILPADRES_encoded',
    'FAMILIARDISCAPACIDAD': 'FAMILIARDISCAPACIDAD_encoded',
    'FAMILIARENFERMEDAD': 'FAMILIARENFERMEDAD_encoded',
    'TIPOPARROQUIA': 'TIPOPARROQUIA_encoded',
    'VIVEGRUPOFAMILIAR': 'VIVEGRUPOFAMILIAR_encoded',
    'SEXO': 'SEXO_encoded',
    'PERDIO_CARRERA': 'PERDIO_CARRERA_encoded',
    'termino': 'termino_encoded'
}

for original_col in encoded_columns.keys():
    if original_col in df_seleccion_final_2.columns and original_col in label_encoders:
        valores_encoder = set(label_encoders[original_col].classes_)
        valores_datos = set(df_seleccion_final_2[original_col].astype(str).unique())
        desconocidos = valores_datos - valores_encoder
        
        if desconocidos:
            print(f"⚠️ {original_col}:")
            print(f"   Valores en encoder: {valores_encoder}")
            print(f"   Valores en datos: {valores_datos}")
            print(f"   DESCONOCIDOS: {desconocidos}\n")
        else:
            print(f"✅ {original_col}: OK\n")

🔍 Verificando valores en cada columna categórica...

✅ TIPOCOLEGIO: OK

✅ BECACOLEGIO: OK

✅ COD_MATERIA_ACAD_MO: OK

✅ TIENEDISCAPACIDAD: OK

✅ TIPODISCAPACIDAD: OK

✅ ESTADOCIVIL: OK

✅ OTROSIDIOMAS: OK

✅ TIEMPOPROMEDIOLLEGARESPOL: OK

✅ NIVELINGLES: OK

✅ NIVELINSTRUCCIONPADRE: OK

✅ NIVELINSTRUCCIONMADRE: OK

✅ ESTADOCIVILPADRES: OK

✅ FAMILIARDISCAPACIDAD: OK

✅ FAMILIARENFERMEDAD: OK

✅ TIPOPARROQUIA: OK

✅ VIVEGRUPOFAMILIAR: OK

✅ SEXO: OK

✅ PERDIO_CARRERA: OK

✅ termino: OK



In [505]:
# Aplicar label encoders a las columnas categóricas
encoded_columns = {
    'TIPOCOLEGIO': 'TIPOCOLEGIO_encoded',
    'BECACOLEGIO': 'BECACOLEGIO_encoded',
    'COD_MATERIA_ACAD_MO': 'COD_MATERIA_ACAD_MO_encoded',
    'TIENEDISCAPACIDAD': 'TIENEDISCAPACIDAD_encoded',
    'TIPODISCAPACIDAD': 'TIPODISCAPACIDAD_encoded',
    'ESTADOCIVIL': 'ESTADOCIVIL_encoded',
    'OTROSIDIOMAS': 'OTROSIDIOMAS_encoded',
    'TIEMPOPROMEDIOLLEGARESPOL': 'TIEMPOPROMEDIOLLEGARESPOL_encoded',
    'NIVELINGLES': 'NIVELINGLES_encoded',
    'NIVELINSTRUCCIONPADRE': 'NIVELINSTRUCCIONPADRE_encoded',
    'NIVELINSTRUCCIONMADRE': 'NIVELINSTRUCCIONMADRE_encoded',
    'ESTADOCIVILPADRES': 'ESTADOCIVILPADRES_encoded',
    'FAMILIARDISCAPACIDAD': 'FAMILIARDISCAPACIDAD_encoded',
    'FAMILIARENFERMEDAD': 'FAMILIARENFERMEDAD_encoded',
    'TIPOPARROQUIA': 'TIPOPARROQUIA_encoded',
    'VIVEGRUPOFAMILIAR': 'VIVEGRUPOFAMILIAR_encoded',
    'SEXO': 'SEXO_encoded',
    'PERDIO_CARRERA': 'PERDIO_CARRERA_encoded',
    'termino': 'termino_encoded'
}

# Aplicar transformación con los label_encoders
for original_col, encoded_col in encoded_columns.items():
    try:
        if original_col in df_seleccion_final_2.columns and original_col in label_encoders:
            print(f"Transformando {original_col}...")
            df_seleccion_final_2[encoded_col] = label_encoders[original_col].transform(
                df_seleccion_final_2[original_col].astype(str)
            )
            print(f"✅ {original_col} → {encoded_col}")
        else:
            print(f"⚠️ {original_col} no encontrada o sin encoder disponible")
    except Exception as e:
        print(f"Error transformando {original_col}: {e}")

Transformando TIPOCOLEGIO...
✅ TIPOCOLEGIO → TIPOCOLEGIO_encoded
Transformando BECACOLEGIO...
✅ BECACOLEGIO → BECACOLEGIO_encoded
Transformando COD_MATERIA_ACAD_MO...
✅ COD_MATERIA_ACAD_MO → COD_MATERIA_ACAD_MO_encoded
Transformando TIENEDISCAPACIDAD...
✅ TIENEDISCAPACIDAD → TIENEDISCAPACIDAD_encoded
Transformando TIPODISCAPACIDAD...
✅ TIPODISCAPACIDAD → TIPODISCAPACIDAD_encoded
Transformando ESTADOCIVIL...
✅ ESTADOCIVIL → ESTADOCIVIL_encoded
Transformando OTROSIDIOMAS...
✅ OTROSIDIOMAS → OTROSIDIOMAS_encoded
Transformando TIEMPOPROMEDIOLLEGARESPOL...
✅ TIEMPOPROMEDIOLLEGARESPOL → TIEMPOPROMEDIOLLEGARESPOL_encoded
Transformando NIVELINGLES...
✅ NIVELINGLES → NIVELINGLES_encoded
Transformando NIVELINSTRUCCIONPADRE...
✅ NIVELINSTRUCCIONPADRE → NIVELINSTRUCCIONPADRE_encoded
Transformando NIVELINSTRUCCIONMADRE...
✅ NIVELINSTRUCCIONMADRE → NIVELINSTRUCCIONMADRE_encoded
Transformando ESTADOCIVILPADRES...
✅ ESTADOCIVILPADRES → ESTADOCIVILPADRES_encoded
Transformando FAMILIARDISCAPACIDAD...
✅ 

In [506]:
df_seleccion_final_2["PROMEDIO_MO"] = df_seleccion_final_2["PROMEDIO_MO"].str.replace(",", ".").astype(float)
df_seleccion_final_2["PROM_CALIFICACIONES"] = df_seleccion_final_2["PROM_CALIFICACIONES"].str.replace(",", ".").astype(float)
df_seleccion_final_2["PROM_CALIF_APROBADAS"] = df_seleccion_final_2["PROM_CALIF_APROBADAS"].str.replace(",", ".").astype(float)
df_seleccion_final_2["PROM_MAT_REPROBADAS2"] = df_seleccion_final_2["PROM_MAT_REPROBADAS2"].str.replace(",", ".").astype(float)
df_seleccion_final_2["PROM_MAT_REPROBADAS3"] = df_seleccion_final_2["PROM_MAT_REPROBADAS3"].str.replace(",", ".").astype(float)

In [507]:
df_seleccion_final_2

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,NIVELINSTRUCCIONPADRE_encoded,NIVELINSTRUCCIONMADRE_encoded,ESTADOCIVILPADRES_encoded,FAMILIARDISCAPACIDAD_encoded,FAMILIARENFERMEDAD_encoded,TIPOPARROQUIA_encoded,VIVEGRUPOFAMILIAR_encoded,SEXO_encoded,PERDIO_CARRERA_encoded,termino_encoded
0,200123669,CCPG1056,AC,2,47,49,4.80,5.72,15.0,58.0,...,2,3,6,297,173,1,1,0,0,0
1,200525939,CCPG1056,AC,1,22,2,1.20,5.72,18.0,64.0,...,0,0,5,297,173,1,1,1,0,0
2,200533511,CCPG1056,AC,2,0,0,0.00,5.72,17.0,50.0,...,5,7,4,297,406,1,1,1,0,1
3,201020203,CCPG1056,AC,1,7,0,0.84,5.72,57.0,56.0,...,9,6,0,297,173,1,0,0,0,1
4,201122213,CCPG1056,AC,2,46,21,3.50,5.72,47.0,64.0,...,0,0,5,297,173,1,1,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223,202408472,CCPG1056,AC,1,0,0,0.00,5.72,13.0,78.0,...,11,12,0,297,173,1,1,0,0,1
224,202408845,CCPG1056,AC,1,0,0,0.00,5.72,15.0,82.0,...,12,12,0,297,173,1,1,1,0,1
225,202409512,CCPG1056,AC,1,0,0,0.00,5.72,16.0,77.0,...,6,11,0,297,173,1,1,1,0,1
226,202409645,CCPG1056,AC,1,0,0,0.00,5.72,14.0,87.0,...,0,11,1,297,173,1,1,1,0,1


In [508]:
feature_info

{'features_numericas': ['VEZ_x_DIFICULTAD',
  'MAT_APROBADAS',
  'PROM_CALIF_APROBADAS',
  'TERMINOS_REGISTRADOS',
  'PROM_MAT_REPROBADAS1',
  'PROM_MAT_REPROBADAS2',
  'PROM_MAT_REPROBADAS3',
  'MUY_FACIL',
  'FACIL',
  'MODERADA',
  'DIFICIL',
  'MUY_DIFICIL',
  'PORCENTAJEDISCAPACIDAD',
  'NUMERO_IDIOMAS',
  'VECESBUSENTRADA',
  'VECESBUSSALIDA',
  'CANTIDADCUARTOS',
  'CANTIDADBANIO',
  'edad_ingreso',
  'RATIO_APROBADAS',
  'TASA_REPROBACION',
  'LOG_CANT_MAT',
  'GPA_CUADRADO',
  'LOG_GASTOS_RUBRO'],
 'features_categoricas': ['TIPOCOLEGIO',
  'BECACOLEGIO',
  'COD_MATERIA_ACAD_MO',
  'TIENEDISCAPACIDAD',
  'TIPODISCAPACIDAD',
  'ESTADOCIVIL',
  'OTROSIDIOMAS',
  'TIEMPOPROMEDIOLLEGARESPOL',
  'NIVELINGLES',
  'NIVELINSTRUCCIONPADRE',
  'NIVELINSTRUCCIONMADRE',
  'ESTADOCIVILPADRES',
  'FAMILIARDISCAPACIDAD',
  'FAMILIARENFERMEDAD',
  'TIPOPARROQUIA',
  'VIVEGRUPOFAMILIAR',
  'SEXO',
  'PERDIO_CARRERA',
  'termino'],
 'X_columns': ['VEZ_x_DIFICULTAD',
  'MAT_APROBADAS',
  'PROM_CA

In [509]:
results = predict_academic_risk(modelo, feature_info, df_seleccion_final_2, return_dataframe=True)
stats = results['statistics']

📊 Usando DataFrame proporcionado
   ✅ Datos cargados: 228 registros
🔮 Realizando predicciones con 228 registros...
** tmp_categoria_riesgo <class 'pandas.core.arrays.categorical.Categorical'>
   ✅ Probabilidades calculadas
   ✅ Predicciones completadas
   📊 144 estudiantes aprobarán (63.16%)
   📊 84 estudiantes reprobarán (36.84%)


c:\Users\saraujo\Documents\Riesgo academico\Codigos_riesgo_academico\planificacion_aprobacion\inference.py:148: FutureWarning: Categorical.to_list is deprecated and will be removed in a future version. Use obj.tolist() instead
  results["CATEGORIA_RIESGO"] = tmp_categoria_riesgo.to_list()


In [510]:
results

{'predictions': [0,
  0,
  0,
  0,
  1,
  1,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  1,
  0,
  0,
  1,
  1,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  1,
  1,
  1,
  0,
  1,
  0,
  0,
  0,
  1,
  0,
  0,
  0,
  0,
  1,
  0,
  1,
  0,
  0,
  0,
  0,
  0,
  0,
  1,
  1,
  0,
  0,
  1,
  1,
  0,
  0,
  1,
  0,
  0,
  1,
  1,
  0,
  1,
  0,
  0,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  0,
  1,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  0,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  1,
  1,
  0,
  1,
  1,
  1,
  1,
  0,
  0,
  1,
  1,
  1,
  1,
  0,
  1,


In [511]:
stats

{'total_estudiantes': 228,
 'pred_aprobar': 144,
 'pred_reprobar': 84,
 'pct_aprobar': 63.1578947368421,
 'pct_reprobar': 36.84210526315789,
 'prob_promedio': 0.6077138818227654,
 'prob_std': 0.21613222029647844,
 'prob_min': 0.17460763639351168,
 'prob_max': 0.9445771398250513,
 'distribucion_riesgo': {'MUY PROBABLE APROBACIÓN': 91,
  'POCO PROBABLE': 66,
  'PROBABLE APROBACIÓN': 53,
  'MUY POCO PROBABLE': 18}}

In [512]:
df_seleccion_final_2.shape, len(results["predictions"])

((228, 85), 228)

### Save dataframe with predictions

In [513]:
df_new_semester = pd.read_csv("../data/riesgo_academico/all_2026_1S.csv")

In [514]:
df_new_semester.keys()

Index(['COD_ESTUDIANTE', 'COD_MATERIA_ACAD_MO', 'ESTADO_MAT_TOMADA_MO',
       'VEZ_TOMADA_MO', 'NOTA1_MO', 'NOTA2MO', 'PROMEDIO_MO', 'DIFICULTAD_MO',
       'T_MAT_TOMADAS', 'PROM_1PARCIAL', 'PROM_2PARCIAL',
       'PROM_CALIFICACIONES', 'MAT_APROBADAS', 'PROM_CALIF_APROBADAS',
       'TERMINOS_REGISTRADOS', 'PERDIO_CARRERA', 'PROM_MAT_REPROBADAS1',
       'PROM_MAT_REPROBADAS2', 'PROM_MAT_REPROBADAS3', 'MUY_FACIL', 'FACIL',
       'MODERADA', 'DIFICIL', 'MUY_DIFICIL', 'promedio_general',
       'CODIGOMATERIA', 'MATERIA'],
      dtype='object')

In [515]:
df_seleccion_final_2[df_new_semester.keys()]

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,PROM_MAT_REPROBADAS2,PROM_MAT_REPROBADAS3,MUY_FACIL,FACIL,MODERADA,DIFICIL,MUY_DIFICIL,promedio_general,CODIGOMATERIA,MATERIA
0,200123669,CCPG1056,AC,2,47,49,4.80,5.72,15.0,58.0,...,NaN,NaN,0,0,0,0,3,NaN,CCPG1056,SISTEMAS OPERATIVOS
1,200525939,CCPG1056,AC,1,22,2,1.20,5.72,18.0,64.0,...,NaN,NaN,0,0,0,0,1,NaN,CCPG1056,SISTEMAS OPERATIVOS
2,200533511,CCPG1056,AC,2,0,0,0.00,5.72,17.0,50.0,...,NaN,NaN,0,0,0,0,1,NaN,CCPG1044,INTELIGENCIA ARTIFICIAL
3,201020203,CCPG1056,AC,1,7,0,0.84,5.72,57.0,56.0,...,5.16,NaN,0,1,0,0,0,NaN,CCPG1053,SEGURIDAD DE LA INFORMACIÓN
4,201122213,CCPG1056,AC,2,46,21,3.50,5.72,47.0,64.0,...,4.65,NaN,0,0,0,0,1,NaN,CCPG1056,SISTEMAS OPERATIVOS
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223,202408472,CCPG1056,AC,1,0,0,0.00,5.72,13.0,78.0,...,NaN,NaN,0,0,0,0,5,NaN,CCPG1034,ESTRUCTURAS DE DATOS
224,202408845,CCPG1056,AC,1,0,0,0.00,5.72,15.0,82.0,...,NaN,NaN,0,0,0,1,5,NaN,CCPG1034,ESTRUCTURAS DE DATOS
225,202409512,CCPG1056,AC,1,0,0,0.00,5.72,16.0,77.0,...,NaN,NaN,0,0,0,1,4,NaN,CCPG1051,PROGRAMACIÓN DE SISTEMAS
226,202409645,CCPG1056,AC,1,0,0,0.00,5.72,14.0,87.0,...,NaN,NaN,0,0,0,0,5,NaN,ESTG1034,ESTADÍSTICA


In [516]:
# df_seleccion_final_2["ESTADO_MAT_TOMADA_MO"] = if results["predictions"] = 0 "RP" else "AP"
df_seleccion_final_2["ESTADO_MAT_TOMADA_MO"] = df_seleccion_final_2.apply(
    lambda row: 'AP' if results["predictions"][row.name] == 1 else 'RP',
    axis=1
)

In [517]:
df_seleccion_final_2

,COD_ESTUDIANTE,COD_MATERIA_ACAD_MO,ESTADO_MAT_TOMADA_MO,VEZ_TOMADA_MO,NOTA1_MO,NOTA2MO,PROMEDIO_MO,DIFICULTAD_MO,T_MAT_TOMADAS,PROM_1PARCIAL,...,NIVELINSTRUCCIONPADRE_encoded,NIVELINSTRUCCIONMADRE_encoded,ESTADOCIVILPADRES_encoded,FAMILIARDISCAPACIDAD_encoded,FAMILIARENFERMEDAD_encoded,TIPOPARROQUIA_encoded,VIVEGRUPOFAMILIAR_encoded,SEXO_encoded,PERDIO_CARRERA_encoded,termino_encoded
0,200123669,CCPG1056,RP,2,47,49,4.80,5.72,15.0,58.0,...,2,3,6,297,173,1,1,0,0,0
1,200525939,CCPG1056,RP,1,22,2,1.20,5.72,18.0,64.0,...,0,0,5,297,173,1,1,1,0,0
2,200533511,CCPG1056,RP,2,0,0,0.00,5.72,17.0,50.0,...,5,7,4,297,406,1,1,1,0,1
3,201020203,CCPG1056,RP,1,7,0,0.84,5.72,57.0,56.0,...,9,6,0,297,173,1,0,0,0,1
4,201122213,CCPG1056,AP,2,46,21,3.50,5.72,47.0,64.0,...,0,0,5,297,173,1,1,1,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
223,202408472,CCPG1056,AP,1,0,0,0.00,5.72,13.0,78.0,...,11,12,0,297,173,1,1,0,0,1
224,202408845,CCPG1056,AP,1,0,0,0.00,5.72,15.0,82.0,...,12,12,0,297,173,1,1,1,0,1
225,202409512,CCPG1056,AP,1,0,0,0.00,5.72,16.0,77.0,...,6,11,0,297,173,1,1,1,0,1
226,202409645,CCPG1056,AP,1,0,0,0.00,5.72,14.0,87.0,...,0,11,1,297,173,1,1,1,0,1


In [518]:
print("Para la materia objetivo:", cod_materia_objetivo, ":", materia_objetivo)

Para la materia objetivo: CCPG1056 : SISTEMAS OPERATIVOS


In [519]:
df_seleccion_final_2["ESTADO_MAT_TOMADA_MO"].value_counts()

ESTADO_MAT_TOMADA_MO
AP    144
RP     84
Name: count, dtype: int64